In [1]:
# ============================================================
# CELL 1 — CLONE OFFICIAL FEW-SHOT KWS REPOSITORY
# ============================================================

%cd /kaggle/working

!git clone https://github.com/ArchitParnami/Few-Shot-KWS.git

%cd /kaggle/working/Few-Shot-KWS

print("\n✅ Repository cloned successfully.")
print("Current directory:")
!pwd

print("\nTop-level files:")
!ls -lah

/kaggle/working
fatal: destination path 'Few-Shot-KWS' already exists and is not an empty directory.
/kaggle/working/Few-Shot-KWS

✅ Repository cloned successfully.
Current directory:
/kaggle/working/Few-Shot-KWS

Top-level files:
total 31M
drwxr-xr-x 10 root root 4.0K Oct  2 08:01 .
drwxr-xr-x  7 root root 4.0K Oct  2 08:03 ..
drwxr-xr-x  3 root root 4.0K Oct  2 08:01 data
drwxr-xr-x  2 root root 4.0K Oct  2 08:01 figures
drwxr-xr-x  8 root root 4.0K Oct  2 08:01 .git
-rw-r--r--  1 root root  109 Oct  2 08:01 .gitignore
drwxr-xr-x  6 root root 4.0K Oct  2 08:01 protonets
drwxr-xr-x  2 root root 4.0K Oct  2 08:01 protonets.egg-info
-rw-r--r--  1 root root 2.8K Oct  2 08:01 README.md
-rw-r--r--  1 root root   37 Oct  2 08:01 requirements.txt
drwxr-xr-x  5 root root 4.0K Oct  2 08:01 results
drwxr-xr-x  4 root root 4.0K Oct  2 08:01 scripts
-rw-r--r--  1 root root  269 Oct  2 08:01 setup.py
drwxr-x---  5 root root 4.0K Oct  2 08:01 speech_commands
-rw-r--r--  1 root root  31M Oct  2 08:0

In [21]:
# ============================================================
# CELL 2 — INSPECT DATA PREPARATION PIPELINE
# ============================================================

from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

print("=" * 80)
print("DATA DIRECTORY")
print("=" * 80)

for p in sorted((REPO / "data").rglob("*")):
    depth = len(p.relative_to(REPO / "data").parts)
    indent = "    " * (depth - 1)
    print(f"{indent}{p.name}")

print("\n" + "=" * 80)
print("SCRIPTS DIRECTORY")
print("=" * 80)

for p in sorted((REPO / "scripts").rglob("*")):
    if p.is_file():
        print(p.relative_to(REPO))


print("\n" + "=" * 80)
print("CANDIDATE DATA PREPARATION FILES")
print("=" * 80)

candidates = []

for p in REPO.rglob("*.py"):
    name = p.name.lower()

    if any(
        key in name
        for key in [
            "download",
            "prepare",
            "dataset",
            "data",
            "preprocess",
        ]
    ):
        candidates.append(p)

for p in sorted(candidates):
    print(p.relative_to(REPO))


print("\n" + "=" * 80)
print("CONTENT OF data/download_prepare_data.py")
print("=" * 80)

target = REPO / "data" / "download_prepare_data.py"

if target.exists():
    print(target.read_text())
else:
    print("❌ data/download_prepare_data.py not found")

DATA DIRECTORY
download_prepare_data.py

SCRIPTS DIRECTORY
scripts/debug_args.py
scripts/parse_results.py
scripts/plot.py
scripts/predict/few_shot/eval.py
scripts/predict/few_shot/eval_results.py
scripts/predict/few_shot/qsub_eval.sh
scripts/predict/few_shot/run_eval.py
scripts/predict/few_shot/submit.sh
scripts/train/few_shot/fewshotspeech/loop_qsub.sh
scripts/train/few_shot/fewshotspeech/qsub_train.sh
scripts/train/few_shot/fewshotspeech/train.sh
scripts/train/few_shot/fewshotspeech/train_fewshotspeech.sh
scripts/train/few_shot/run_train.py
scripts/train/few_shot/run_trainval.py
scripts/train/few_shot/train.py
scripts/train/few_shot/trainval.py

CANDIDATE DATA PREPARATION FILES
data/download_prepare_data.py
protonets/data/FewShotSpeechData.py
protonets/utils/data.py

CONTENT OF data/download_prepare_data.py
import os
import sys
import urllib
import tarfile
import torchaudio
import glob
import random
import shutil


class FewShotSpeechDataDownloader(object):
    def __init__(self, dat

In [2]:
# ============================================================
# CELL 3A — FIX BUG IN PAPER DOWNLOAD SCRIPT
# ============================================================

from pathlib import Path

script_path = Path(
    "/kaggle/working/Few-Shot-KWS/data/download_prepare_data.py"
)

text = script_path.read_text()

old = "urllib.request.urlretrieve(data_url, filepath, _progress)"
new = "urllib.request.urlretrieve(self.data_url, filepath, _progress)"

if old in text:
    text = text.replace(old, new)
    script_path.write_text(text)
    print("✅ Fixed paper bug:")
    print(old)
    print("→")
    print(new)
else:
    print("ℹ️ Bug already fixed or line not found.")

ℹ️ Bug already fixed or line not found.


In [3]:
# ============================================================
# CELL 3B-FAST — FAST DOWNLOAD RAW GSC v0.02
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import os
from pathlib import Path

dataset_dir = Path("/kaggle/working/Few-Shot-KWS/speech_commands")
dataset_dir.mkdir(parents=True, exist_ok=True)

archive = dataset_dir / "speech_commands_v0.02.tar.gz"

url = "https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz"

print("Downloading archive with wget...")

!wget -c -q --show-progress \
    "{url}" \
    -O "{archive}"

print("\nDownload finished.")
print("Archive size:",
      round(archive.stat().st_size / (1024**3), 3),
      "GB")

/kaggle/working/Few-Shot-KWS

Download finished.
Archive size: 2.262 GB


In [4]:
# ============================================================
# CELL 3C — EXTRACT RAW GSC
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import tarfile
from pathlib import Path

dataset_dir = Path("/kaggle/working/Few-Shot-KWS/speech_commands")
archive = dataset_dir / "speech_commands_v0.02.tar.gz"

print("Extracting...")

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(dataset_dir)

print("✅ Extraction complete.")

/kaggle/working/Few-Shot-KWS
Extracting...


/tmp/ipykernel_57/3360490402.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dataset_dir)


✅ Extraction complete.


In [25]:
from pathlib import Path

dataset_dir = Path("/kaggle/working/Few-Shot-KWS/speech_commands")

wav_files = list(dataset_dir.rglob("*.wav"))

print("Total WAV:", len(wav_files))

print("\nTop-level entries:")
for p in sorted(dataset_dir.iterdir())[:50]:
    print(p.name)

Total WAV: 105835

Top-level entries:
.DS_Store
LICENSE
README.md
_background_noise_
backward
bed
bird
cat
dog
down
eight
five
follow
forward
four
go
happy
house
learn
left
marvin
nine
no
off
on
one
right
seven
sheila
six
speech_commands_v0.02.tar.gz
stop
testing_list.txt
three
tree
two
up
validation_list.txt
visual
wow
yes
zero


In [26]:
# ============================================================
# CELL 4A — RAW GSC AUDIT
# ============================================================

from pathlib import Path
from collections import Counter
import soundfile as sf
import pandas as pd
import numpy as np

GSC_ROOT = Path("/kaggle/working/Few-Shot-KWS/speech_commands")

# ------------------------------------------------------------
# 1. Collect WAV files
# ------------------------------------------------------------
wav_files = list(GSC_ROOT.rglob("*.wav"))

print("=" * 80)
print("RAW GSC AUDIT")
print("=" * 80)

print("Total WAV files:", len(wav_files))


# ------------------------------------------------------------
# 2. Class distribution
# ------------------------------------------------------------
special = {"_background_noise_"}

keyword_files = [
    p for p in wav_files
    if p.parent.name not in special
]

class_counts = Counter(p.parent.name for p in keyword_files)

print("Keyword classes:", len(class_counts))

print("\nTop classes by sample count:")
for label, count in class_counts.most_common(40):
    print(f"{label:12s} {count}")


# ------------------------------------------------------------
# 3. Fast metadata audit
#    Read ALL files but metadata only, not waveform
# ------------------------------------------------------------
rows = []
errors = []

for i, p in enumerate(wav_files):
    try:
        info = sf.info(p)

        rows.append({
            "path": str(p),
            "label": p.parent.name,
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "frames": info.frames,
            "duration_sec": (
                info.frames / info.samplerate
                if info.samplerate > 0
                else np.nan
            ),
            "format": info.format,
            "subtype": info.subtype,
        })

    except Exception as e:
        errors.append({
            "path": str(p),
            "error": repr(e)
        })


df = pd.DataFrame(rows)
error_df = pd.DataFrame(errors)


# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("METADATA SUMMARY")
print("=" * 80)

print("Readable files:", len(df))
print("Errors:", len(error_df))

print("\nSample rate distribution:")
print(df["sample_rate"].value_counts().sort_index())

print("\nChannel distribution:")
print(df["channels"].value_counts().sort_index())

print("\nFormat distribution:")
print(df["format"].value_counts())

print("\nSubtype distribution:")
print(df["subtype"].value_counts())

print("\nDuration statistics:")
print(df["duration_sec"].describe())


# ------------------------------------------------------------
# 5. Count files shorter than 1 second
# ------------------------------------------------------------
short_df = df[df["duration_sec"] < 1.0]

print("\n" + "=" * 80)
print("SHORT AUDIO CHECK")
print("=" * 80)

print("Files < 1.0 sec:", len(short_df))
print(
    "Percent < 1.0 sec:",
    round(len(short_df) / len(df) * 100, 4),
    "%"
)

if len(short_df):
    print("\nShortest 20 files:")
    print(
        short_df[
            ["label", "duration_sec", "path"]
        ]
        .sort_values("duration_sec")
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 6. Background noise
# ------------------------------------------------------------
bg_df = df[df["label"] == "_background_noise_"]

print("\n" + "=" * 80)
print("BACKGROUND NOISE")
print("=" * 80)

print("Background WAV files:", len(bg_df))

if len(bg_df):
    print(bg_df["duration_sec"].describe())


# ------------------------------------------------------------
# 7. Save report
# ------------------------------------------------------------
REPORT_DIR = Path("/kaggle/working/reports/data_audit")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    REPORT_DIR / "gsc_raw_metadata.csv",
    index=False
)

pd.DataFrame(
    sorted(class_counts.items()),
    columns=["class", "num_samples"]
).to_csv(
    REPORT_DIR / "gsc_raw_class_distribution.csv",
    index=False
)

if len(error_df):
    error_df.to_csv(
        REPORT_DIR / "gsc_raw_errors.csv",
        index=False
    )

short_df.to_csv(
    REPORT_DIR / "gsc_raw_shorter_than_1s.csv",
    index=False
)

print("\nSaved audit files to:")
print(REPORT_DIR)

RAW GSC AUDIT
Total WAV files: 105835
Keyword classes: 35

Top classes by sample count:
zero         4052
five         4052
yes          4044
seven        3998
no           3941
nine         3934
down         3917
one          3890
go           3880
two          3880
stop         3872
six          3860
on           3845
left         3801
eight        3787
right        3778
off          3745
four         3728
three        3727
up           3723
dog          2128
wow          2123
house        2113
marvin       2100
bird         2064
happy        2054
cat          2031
sheila       2022
bed          2014
tree         1759
backward     1664
visual       1592
follow       1579
learn        1575
forward      1557

METADATA SUMMARY
Readable files: 105835
Errors: 0

Sample rate distribution:
sample_rate
16000    105835
Name: count, dtype: int64

Channel distribution:
channels
1    105835
Name: count, dtype: int64

Format distribution:
format
WAV    105835
Name: count, dtype: int64

Subtype di

In [27]:
# ============================================================
# CELL 4B — SPEAKER AUDIT FOR PAPER PIPELINE
# ============================================================

from pathlib import Path
from collections import defaultdict
import pandas as pd

GSC_ROOT = Path("/kaggle/working/Few-Shot-KWS/speech_commands")

speaker_map = defaultdict(set)

for class_dir in GSC_ROOT.iterdir():
    if not class_dir.is_dir():
        continue

    if class_dir.name.startswith("_"):
        continue

    for wav in class_dir.glob("*.wav"):
        name = wav.name

        if "_nohash_" not in name:
            continue

        speaker_id = name.split("_nohash_")[0]

        speaker_map[class_dir.name].add(speaker_id)


rows = []

for keyword, speakers in speaker_map.items():
    rows.append({
        "keyword": keyword,
        "num_speakers": len(speakers),
        "core_if_limit_1000": len(speakers) >= 1000
    })

speaker_df = pd.DataFrame(rows).sort_values(
    "num_speakers",
    ascending=False
)

print("=" * 80)
print("SPEAKERS PER KEYWORD")
print("=" * 80)

print(speaker_df.to_string(index=False))

core_df = speaker_df[
    speaker_df["core_if_limit_1000"]
]

unknown_df = speaker_df[
    ~speaker_df["core_if_limit_1000"]
]

print("\nCore words:", len(core_df))
print(core_df["keyword"].tolist())

print("\nUnknown words:", len(unknown_df))
print(unknown_df["keyword"].tolist())

speaker_df.to_csv(
    "/kaggle/working/reports/data_audit/gsc_raw_speaker_distribution.csv",
    index=False
)

SPEAKERS PER KEYWORD
 keyword  num_speakers  core_if_limit_1000
    down          1668                True
   seven          1656                True
     yes          1646                True
    nine          1645                True
    four          1631                True
    five          1624                True
      go          1612                True
    left          1611                True
     one          1608                True
   right          1606                True
      on          1603                True
    zero          1602                True
      no          1590                True
   three          1586                True
    stop          1586                True
     six          1583                True
   eight          1566                True
     dog          1565                True
     off          1564                True
     two          1562                True
  marvin          1557                True
      up          1547           

In [28]:
# ============================================================
# AUDIOMNIST ULTRA-FAST STRUCTURE AUDIT
# ============================================================

import os
from collections import Counter
from pathlib import Path

ROOT = "/kaggle/input/datasets/sripaadsrinivasan/audio-mnist/data"

speaker_dirs = sorted(os.listdir(ROOT))

digit_counts = Counter()
speaker_counts = Counter()
total = 0

for speaker in speaker_dirs:
    speaker_path = os.path.join(ROOT, speaker)

    if not os.path.isdir(speaker_path):
        continue

    files = os.listdir(speaker_path)

    for name in files:
        if not name.endswith(".wav"):
            continue

        total += 1

        parts = Path(name).stem.split("_")

        if len(parts) >= 3:
            digit = parts[0]
            speaker_id = parts[1]

            digit_counts[digit] += 1
            speaker_counts[speaker_id] += 1

print("Total WAV:", total)
print("Digits:", len(digit_counts))
print("Speakers:", len(speaker_counts))
print("Digit distribution:", dict(sorted(digit_counts.items())))

Total WAV: 30000
Digits: 10
Speakers: 60
Digit distribution: {'0': 3000, '1': 3000, '2': 3000, '3': 3000, '4': 3000, '5': 3000, '6': 3000, '7': 3000, '8': 3000, '9': 3000}


In [29]:
import soundfile as sf
import os

samples = []

for speaker in speaker_dirs[:2]:
    folder = os.path.join(ROOT, speaker)

    for name in os.listdir(folder):
        if name.endswith(".wav"):
            samples.append(os.path.join(folder, name))

        if len(samples) >= 10:
            break

    if len(samples) >= 10:
        break

for p in samples:
    info = sf.info(p)
    print(
        os.path.basename(p),
        "sr=", info.samplerate,
        "ch=", info.channels,
        "dur=", round(info.frames / info.samplerate, 3),
        "fmt=", info.format,
        info.subtype
    )

6_01_49.wav sr= 48000 ch= 1 dur= 0.727 fmt= WAV PCM_16
1_01_26.wav sr= 48000 ch= 1 dur= 0.431 fmt= WAV PCM_16
3_01_32.wav sr= 48000 ch= 1 dur= 0.637 fmt= WAV PCM_16
3_01_12.wav sr= 48000 ch= 1 dur= 0.565 fmt= WAV PCM_16
4_01_17.wav sr= 48000 ch= 1 dur= 0.597 fmt= WAV PCM_16
8_01_10.wav sr= 48000 ch= 1 dur= 0.593 fmt= WAV PCM_16
6_01_30.wav sr= 48000 ch= 1 dur= 0.672 fmt= WAV PCM_16
6_01_14.wav sr= 48000 ch= 1 dur= 0.823 fmt= WAV PCM_16
9_01_32.wav sr= 48000 ch= 1 dur= 0.644 fmt= WAV PCM_16
3_01_25.wav sr= 48000 ch= 1 dur= 0.653 fmt= WAV PCM_16


In [30]:
# ============================================================
# CELL 6A — FLUENT SPEECH COMMANDS METADATA AUDIT
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/kaggle/input/datasets/muhammadnaseem2020/"
    "fluent-speech-commands/fluent_speech_commands_dataset"
)

DATA_DIR = ROOT / "data"

# Load official dataset splits
train_df = pd.read_csv(DATA_DIR / "train_data.csv")
valid_df = pd.read_csv(DATA_DIR / "valid_data.csv")
test_df  = pd.read_csv(DATA_DIR / "test_data.csv")

train_df["split"] = "train"
valid_df["split"] = "valid"
test_df["split"] = "test"

df_fsc = pd.concat(
    [train_df, valid_df, test_df],
    ignore_index=True
)

print("=" * 80)
print("FLUENT SPEECH COMMANDS — METADATA AUDIT")
print("=" * 80)

print("Train samples :", len(train_df))
print("Valid samples :", len(valid_df))
print("Test samples  :", len(test_df))
print("Total samples :", len(df_fsc))

print("\nColumns:")
print(df_fsc.columns.tolist())


# ------------------------------------------------------------
# Detect transcript column automatically
# ------------------------------------------------------------
if "transcription" in df_fsc.columns:
    TEXT_COL = "transcription"
elif "transcript" in df_fsc.columns:
    TEXT_COL = "transcript"
else:
    TEXT_COL = None


# ------------------------------------------------------------
# Speakers
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("SPEAKERS")
print("=" * 80)

print("Unique speakers:", df_fsc["speakerId"].nunique())


# ------------------------------------------------------------
# Spoken phrases
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("PHRASES")
print("=" * 80)

if TEXT_COL:
    print("Transcript column:", TEXT_COL)
    print("Unique phrases:", df_fsc[TEXT_COL].nunique())

    print("\nExample phrases:")
    for text in df_fsc[TEXT_COL].drop_duplicates().head(15):
        print("-", text)
else:
    print("Transcript column not detected.")


# ------------------------------------------------------------
# Semantic slots
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("ACTION")
print("=" * 80)

print("Unique actions:", df_fsc["action"].nunique())
print(sorted(df_fsc["action"].dropna().unique()))


print("\n" + "=" * 80)
print("OBJECT")
print("=" * 80)

print("Unique objects:", df_fsc["object"].nunique())
print(sorted(df_fsc["object"].dropna().unique()))


print("\n" + "=" * 80)
print("LOCATION")
print("=" * 80)

print("Unique locations:", df_fsc["location"].nunique())
print(sorted(df_fsc["location"].dropna().unique()))


# ------------------------------------------------------------
# Intent = action + object + location
# ------------------------------------------------------------
df_fsc["intent"] = (
    df_fsc["action"].fillna("none").astype(str)
    + " | "
    + df_fsc["object"].fillna("none").astype(str)
    + " | "
    + df_fsc["location"].fillna("none").astype(str)
)

print("\n" + "=" * 80)
print("INTENTS")
print("=" * 80)

print("Unique intents:", df_fsc["intent"].nunique())


# ------------------------------------------------------------
# Split sizes
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("SPLITS")
print("=" * 80)

print(df_fsc["split"].value_counts())


# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("SAMPLE ROWS")
print("=" * 80)

cols = [
    "path",
    "speakerId",
    "action",
    "object",
    "location",
    "split"
]

if TEXT_COL:
    cols.insert(2, TEXT_COL)

print(
    df_fsc[cols]
    .head(10)
    .to_string(index=False)
)

FLUENT SPEECH COMMANDS — METADATA AUDIT
Train samples : 23132
Valid samples : 3118
Test samples  : 3793
Total samples : 30043

Columns:
['Unnamed: 0', 'path', 'speakerId', 'transcription', 'action', 'object', 'location', 'split']

SPEAKERS
Unique speakers: 97

PHRASES
Transcript column: transcription
Unique phrases: 248

Example phrases:
- Change language
- Resume
- Turn the lights on
- Switch on the lights
- Switch off the lights
- Volume up
- Turn the volume up
- Turn the volume down
- Turn up the temperature
- Turn the heat up
- Turn down the temperature
- Turn the temperature down
- Turn down the heat
- Turn the heat down
- Turn off the music

ACTION
Unique actions: 6
['activate', 'bring', 'change language', 'deactivate', 'decrease', 'increase']

OBJECT
Unique objects: 14
['Chinese', 'English', 'German', 'Korean', 'heat', 'juice', 'lamp', 'lights', 'music', 'newspaper', 'none', 'shoes', 'socks', 'volume']

LOCATION
Unique locations: 4
['bedroom', 'kitchen', 'none', 'washroom']

INT

In [31]:
# ============================================================
# CELL 6B — FSC AUDIO METADATA SAMPLE
# ============================================================

import soundfile as sf

sample_rows = df_fsc.head(10)

print("=" * 80)
print("FSC AUDIO METADATA SAMPLE")
print("=" * 80)

for _, row in sample_rows.iterrows():
    audio_path = ROOT / row["path"]

    info = sf.info(audio_path)

    duration = info.frames / info.samplerate

    print(
        audio_path.name,
        "sr=", info.samplerate,
        "ch=", info.channels,
        "dur=", round(duration, 3),
        "fmt=", info.format,
        info.subtype
    )

FSC AUDIO METADATA SAMPLE
0a3129c0-4474-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 1.858 fmt= WAV PCM_16
0ee42a80-4474-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 1.393 fmt= WAV PCM_16
144d5be0-4474-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 2.136 fmt= WAV PCM_16
1811b6e0-4474-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 2.229 fmt= WAV PCM_16
1d9f3920-4474-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 1.95 fmt= WAV PCM_16
269fc210-4474-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 1.765 fmt= WAV PCM_16
5bbda3f0-4478-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 2.415 fmt= WAV PCM_16
6436ad60-4478-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 2.229 fmt= WAV PCM_16
6a1cd6f0-4478-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 1.95 fmt= WAV PCM_16
72160200-4478-11e9-a9a5-5dbec3b8816a.wav sr= 16000 ch= 1 dur= 2.043 fmt= WAV PCM_16


In [32]:
# ============================================================
# CELL 7 — PREPROCESS AUDIOMNIST WITH PROGRESS BAR
# ============================================================

from pathlib import Path
import soundfile as sf
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from math import gcd
from tqdm.auto import tqdm

SRC_ROOT = Path(
    "/kaggle/input/datasets/sripaadsrinivasan/audio-mnist/data"
)

OUT_ROOT = Path(
    "/kaggle/working/processed/audiomnist_16k_1s"
)

OUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000
TARGET_SAMPLES = 16000


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def to_mono(audio):
    if audio.ndim == 1:
        return audio

    return audio.mean(axis=1)


def resample_audio(audio, orig_sr, target_sr):
    if orig_sr == target_sr:
        return audio

    g = gcd(orig_sr, target_sr)

    up = target_sr // g
    down = orig_sr // g

    return resample_poly(audio, up, down)


def peak_normalize(audio):
    peak = np.max(np.abs(audio))

    if peak > 0:
        audio = audio / peak

    return audio


def fix_length(audio, target_length=16000):
    length = len(audio)

    # shorter -> center zero-padding
    if length < target_length:
        total_pad = target_length - length

        left = total_pad // 2
        right = total_pad - left

        audio = np.pad(
            audio,
            (left, right),
            mode="constant"
        )

    # longer -> center crop
    elif length > target_length:
        start = (length - target_length) // 2

        audio = audio[
            start:start + target_length
        ]

    return audio


# ------------------------------------------------------------
# Collect file list first
# ------------------------------------------------------------
speaker_dirs = sorted(
    p for p in SRC_ROOT.iterdir()
    if p.is_dir()
)

wav_files = []

for speaker_dir in speaker_dirs:
    wav_files.extend(
        sorted(speaker_dir.glob("*.wav"))
    )

print("=" * 70)
print("AUDIOMNIST PREPROCESSING")
print("=" * 70)

print("Input files :", len(wav_files))
print("Target SR   :", TARGET_SR)
print("Target size :", TARGET_SAMPLES)
print("Output      :", OUT_ROOT)
print()


# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
records = []
errors = []

progress = tqdm(
    wav_files,
    total=len(wav_files),
    desc="AudioMNIST",
    unit="file",
    dynamic_ncols=True
)

for wav_path in progress:

    try:
        audio, sr = sf.read(
            wav_path,
            dtype="float32"
        )

        original_samples = len(audio)
        original_duration = len(audio) / sr

        # 1. mono
        audio = to_mono(audio)

        # 2. resample -> 16 kHz
        audio = resample_audio(
            audio,
            sr,
            TARGET_SR
        )

        # 3. normalize amplitude
        audio = peak_normalize(audio)

        # 4. pad/crop -> exactly 16000 samples
        audio = fix_length(
            audio,
            TARGET_SAMPLES
        )

        speaker_id = wav_path.parent.name

        out_speaker = OUT_ROOT / speaker_id
        out_speaker.mkdir(
            parents=True,
            exist_ok=True
        )

        output_path = (
            out_speaker /
            wav_path.name
        )

        sf.write(
            output_path,
            audio,
            TARGET_SR,
            subtype="PCM_16"
        )

        parts = wav_path.stem.split("_")

        digit = parts[0]

        records.append({
            "original_path": str(wav_path),
            "processed_path": str(output_path),

            "digit": digit,
            "speaker_id": speaker_id,

            "original_sr": sr,
            "target_sr": TARGET_SR,

            "original_samples": original_samples,
            "processed_samples": len(audio),

            "original_duration": original_duration,
            "processed_duration": len(audio) / TARGET_SR
        })

        # tqdm postfix: chỉ cập nhật trên cùng một dòng
        if len(records) % 100 == 0:
            progress.set_postfix(
                ok=len(records),
                err=len(errors)
            )

    except Exception as e:

        errors.append({
            "path": str(wav_path),
            "error": repr(e)
        })


# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------
metadata = pd.DataFrame(records)

metadata.to_csv(
    OUT_ROOT / "metadata.csv",
    index=False
)

if errors:
    pd.DataFrame(errors).to_csv(
        OUT_ROOT / "errors.csv",
        index=False
    )


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("AUDIOMNIST PREPROCESSING COMPLETE")
print("=" * 70)

print("Processed :", len(metadata))
print("Errors    :", len(errors))
print("Output    :", OUT_ROOT)

print("\nProcessed sample rate:")
print(metadata["target_sr"].value_counts())

print("\nProcessed samples/file:")
print(metadata["processed_samples"].value_counts())

print("\nProcessed duration:")
print(metadata["processed_duration"].value_counts())

print("\nDigit distribution:")
print(
    metadata["digit"]
    .value_counts()
    .sort_index()
)

AUDIOMNIST PREPROCESSING
Input files : 30000
Target SR   : 16000
Target size : 16000
Output      : /kaggle/working/processed/audiomnist_16k_1s



AudioMNIST:   0%|          | 0/30000 [00:00<?, ?file/s]


AUDIOMNIST PREPROCESSING COMPLETE
Processed : 30000
Errors    : 0
Output    : /kaggle/working/processed/audiomnist_16k_1s

Processed sample rate:
target_sr
16000    30000
Name: count, dtype: int64

Processed samples/file:
processed_samples
16000    30000
Name: count, dtype: int64

Processed duration:
processed_duration
1.0    30000
Name: count, dtype: int64

Digit distribution:
digit
0    3000
1    3000
2    3000
3    3000
4    3000
5    3000
6    3000
7    3000
8    3000
9    3000
Name: count, dtype: int64


In [33]:
# ============================================================
# CELL 8 — PREPROCESS FSC WITH PROGRESS BAR
# ============================================================

from pathlib import Path
import soundfile as sf
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from math import gcd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
FSC_ROOT = Path(
    "/kaggle/input/datasets/muhammadnaseem2020/"
    "fluent-speech-commands/"
    "fluent_speech_commands_dataset"
)

OUT_ROOT = Path(
    "/kaggle/working/processed/fsc_segments_16k_1s"
)

OUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000
SEGMENT_SAMPLES = 16000


# ------------------------------------------------------------
# Load FSC metadata
# ------------------------------------------------------------
DATA_DIR = FSC_ROOT / "data"

train_df = pd.read_csv(DATA_DIR / "train_data.csv")
valid_df = pd.read_csv(DATA_DIR / "valid_data.csv")
test_df  = pd.read_csv(DATA_DIR / "test_data.csv")

train_df["split"] = "train"
valid_df["split"] = "valid"
test_df["split"] = "test"

df_fsc = pd.concat(
    [train_df, valid_df, test_df],
    ignore_index=True
)

print("=" * 70)
print("FSC PREPROCESSING")
print("=" * 70)

print("Original utterances :", len(df_fsc))
print("Target sample rate  :", TARGET_SR)
print("Segment size        :", SEGMENT_SAMPLES)
print("Output              :", OUT_ROOT)
print()


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def to_mono(audio):
    if audio.ndim == 1:
        return audio

    return audio.mean(axis=1)


def resample_audio(audio, orig_sr, target_sr):
    if orig_sr == target_sr:
        return audio

    g = gcd(orig_sr, target_sr)

    up = target_sr // g
    down = orig_sr // g

    return resample_poly(audio, up, down)


def peak_normalize(audio):
    peak = np.max(np.abs(audio))

    if peak > 0:
        audio = audio / peak

    return audio


def split_into_segments(audio, segment_samples=16000):
    """
    Split full utterance into consecutive 1-second segments.

    Last segment:
    - keep remaining audio
    - zero-pad to exactly segment_samples
    """

    segments = []

    total_samples = len(audio)

    for start in range(0, total_samples, segment_samples):

        segment = audio[
            start:start + segment_samples
        ]

        valid_samples = len(segment)

        if valid_samples < segment_samples:
            segment = np.pad(
                segment,
                (0, segment_samples - valid_samples),
                mode="constant"
            )

        segments.append(
            (
                segment,
                valid_samples
            )
        )

    # Safety for empty waveform
    if len(segments) == 0:
        segments.append(
            (
                np.zeros(segment_samples, dtype=np.float32),
                0
            )
        )

    return segments


# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
records = []
utterance_records = []
errors = []

progress = tqdm(
    df_fsc.itertuples(index=True),
    total=len(df_fsc),
    desc="FSC",
    unit="utt",
    dynamic_ncols=True
)

for row in progress:

    try:

        audio_path = FSC_ROOT / row.path

        audio, sr = sf.read(
            audio_path,
            dtype="float32"
        )

        original_samples = len(audio)
        original_duration = (
            original_samples / sr
            if sr > 0 else 0
        )

        # 1. Mono
        audio = to_mono(audio)

        # 2. Resample if needed
        audio = resample_audio(
            audio,
            sr,
            TARGET_SR
        )

        # 3. Peak normalization
        audio = peak_normalize(audio)

        processed_samples_before_segment = len(audio)

        # 4. Segment
        segments = split_into_segments(
            audio,
            SEGMENT_SAMPLES
        )

        utterance_id = Path(row.path).stem

        # Keep speaker directory structure
        utterance_dir = (
            OUT_ROOT /
            row.speakerId /
            utterance_id
        )

        utterance_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        for segment_id, (segment, valid_samples) in enumerate(segments):

            segment_path = (
                utterance_dir /
                f"segment_{segment_id:03d}.wav"
            )

            sf.write(
                segment_path,
                segment,
                TARGET_SR,
                subtype="PCM_16"
            )

            records.append({
                "utterance_id": utterance_id,
                "segment_id": segment_id,

                "segment_path": str(segment_path),

                "speakerId": row.speakerId,

                "transcription": row.transcription,
                "action": row.action,
                "object": row.object,
                "location": row.location,

                "split": row.split,

                "original_sr": sr,
                "target_sr": TARGET_SR,

                "valid_samples": valid_samples,
                "stored_samples": len(segment),

                "valid_duration_sec":
                    valid_samples / TARGET_SR,

                "stored_duration_sec":
                    len(segment) / TARGET_SR
            })

        utterance_records.append({
            "utterance_id": utterance_id,

            "speakerId": row.speakerId,
            "transcription": row.transcription,

            "action": row.action,
            "object": row.object,
            "location": row.location,

            "split": row.split,

            "original_sr": sr,

            "original_samples": original_samples,
            "original_duration_sec": original_duration,

            "processed_samples":
                processed_samples_before_segment,

            "processed_duration_sec":
                processed_samples_before_segment / TARGET_SR,

            "num_segments": len(segments)
        })

        if len(utterance_records) % 100 == 0:
            progress.set_postfix(
                utt=len(utterance_records),
                seg=len(records),
                err=len(errors)
            )

    except Exception as e:

        errors.append({
            "row_index": row.Index,
            "path": row.path,
            "error": repr(e)
        })


# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------
segment_df = pd.DataFrame(records)

utterance_df = pd.DataFrame(
    utterance_records
)

segment_df.to_csv(
    OUT_ROOT / "segments_metadata.csv",
    index=False
)

utterance_df.to_csv(
    OUT_ROOT / "utterances_metadata.csv",
    index=False
)

if errors:
    pd.DataFrame(errors).to_csv(
        OUT_ROOT / "errors.csv",
        index=False
    )


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FSC PREPROCESSING COMPLETE")
print("=" * 70)

print(
    "Original utterances :",
    len(df_fsc)
)

print(
    "Processed utterances:",
    len(utterance_df)
)

print(
    "Generated segments  :",
    len(segment_df)
)

print(
    "Errors              :",
    len(errors)
)

print(
    "Average segments/utt:",
    round(
        len(segment_df) / len(utterance_df),
        4
    )
)

print("\nTarget sample rate:")
print(
    segment_df["target_sr"]
    .value_counts()
)

print("\nStored samples per segment:")
print(
    segment_df["stored_samples"]
    .value_counts()
)

print("\nStored duration per segment:")
print(
    segment_df["stored_duration_sec"]
    .value_counts()
)

print("\nSegments per utterance:")
print(
    utterance_df["num_segments"]
    .describe()
)

print("\nSplit distribution:")
print(
    utterance_df["split"]
    .value_counts()
)

print("\nOutput:")
print(OUT_ROOT)

FSC PREPROCESSING
Original utterances : 30043
Target sample rate  : 16000
Segment size        : 16000
Output              : /kaggle/working/processed/fsc_segments_16k_1s



FSC:   0%|          | 0/30043 [00:00<?, ?utt/s]


FSC PREPROCESSING COMPLETE
Original utterances : 30043
Processed utterances: 30043
Generated segments  : 83777
Errors              : 0
Average segments/utt: 2.7886

Target sample rate:
target_sr
16000    83777
Name: count, dtype: int64

Stored samples per segment:
stored_samples
16000    83777
Name: count, dtype: int64

Stored duration per segment:
stored_duration_sec
1.0    83777
Name: count, dtype: int64

Segments per utterance:
count    30043.000000
mean         2.788570
std          0.703946
min          1.000000
25%          2.000000
50%          3.000000
75%          3.000000
max         14.000000
Name: num_segments, dtype: float64

Split distribution:
split
train    23132
test      3793
valid     3118
Name: count, dtype: int64

Output:
/kaggle/working/processed/fsc_segments_16k_1s


In [5]:
# ============================================================
# CELL 9 — RUN ORIGINAL FS-GSC PREPARATION PIPELINE
# FIXED FOR CURRENT KAGGLE / TORCHAUDIO
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import time
import random
import importlib.util
from pathlib import Path

import torchaudio


# ------------------------------------------------------------
# 1. LOAD PAPER SCRIPT DIRECTLY
# ------------------------------------------------------------
SCRIPT_PATH = Path(
    "/kaggle/working/Few-Shot-KWS/data/download_prepare_data.py"
)

print("Paper script exists:", SCRIPT_PATH.exists())

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy paper script: {SCRIPT_PATH}"
    )

spec = importlib.util.spec_from_file_location(
    "download_prepare_data",
    SCRIPT_PATH
)

dp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dp)

FewShotSpeechDataDownloader = dp.FewShotSpeechDataDownloader

print("✅ Paper module loaded successfully.")


# ------------------------------------------------------------
# 2. PATCH OLD TORCHAUDIO API
# ------------------------------------------------------------
# Original paper code uses:
#
# torchaudio.load(
#     filepath=audio_file,
#     normalization=True,
#     num_frames=desired_samples
# )
#
# Current Kaggle torchaudio no longer accepts the old API.
# We preserve the same logic:
# read at most desired_samples and return waveform + sample rate.

def load_audio_fixed(self, audio_file, desired_samples):

    sound, sr = torchaudio.load(
        audio_file,
        num_frames=desired_samples
    )

    return sound, sr


FewShotSpeechDataDownloader.load_audio = load_audio_fixed

print("✅ Patched old torchaudio.load() API")


# ------------------------------------------------------------
# 3. CONFIG — SAME AS PAPER REPOSITORY
# ------------------------------------------------------------
DATASET_PATH = (
    "/kaggle/working/Few-Shot-KWS/speech_commands"
)

random.seed(42)

downloader = FewShotSpeechDataDownloader(
    data_url=None,                  # dataset already downloaded/extracted
    dest_directory=DATASET_PATH,

    sample_rate=16000,
    clip_duration_ms=1000,

    speaker_limit=1000,

    core_split=(24, 5, 5),
    unknown_split=(60, 20, 20)
)


# ------------------------------------------------------------
# 4. HELPER TIMER
# ------------------------------------------------------------
def run_step(name, func):

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    start = time.time()

    func()

    elapsed = time.time() - start

    print(f"\n✅ Finished: {name}")
    print(f"Elapsed: {elapsed / 60:.2f} min")


# ------------------------------------------------------------
# 5. VERIFY DATASET EXISTS
# ------------------------------------------------------------
root = Path(DATASET_PATH)

if not root.exists():
    raise FileNotFoundError(
        f"Dataset path không tồn tại: {root}"
    )


keyword_dirs = [
    p for p in root.iterdir()
    if p.is_dir()
    and not p.name.startswith("_")
]

print("\nDataset path:")
print(root)

print("\nKeyword folders:")
print(len(keyword_dirs))


# ------------------------------------------------------------
# 6. VERIFY RAW DATASET STATE
# ------------------------------------------------------------
# RAW structure expected:
#
# speech_commands/
#     yes/
#         xxx_nohash_0.wav
#
# If pipeline already passed grouping:
#
# yes/
#     speaker_id/
#         xxx.wav
#
# then we should not rerun from Step 1.

direct_wav_count = 0

for keyword_dir in keyword_dirs:
    direct_wav_count += sum(
        1 for _ in keyword_dir.glob("*.wav")
    )

print(
    "Direct WAV files inside keyword folders:",
    direct_wav_count
)

if direct_wav_count == 0:
    raise RuntimeError(
        "Dataset không còn ở trạng thái RAW GSC. "
        "Có thể pipeline đã chạy qua bước grouping."
    )


# ============================================================
# STEP 1/6 — FILTERING
# ============================================================
# Delete keyword WAVs shorter than 16000 samples / 1 second.

run_step(
    "STEP 1/6 — FILTERING (<1s REMOVED)",
    downloader.delete_smaller_files
)


# ============================================================
# STEP 2/6 — GROUPING BY SPEAKER
# ============================================================

run_step(
    "STEP 2/6 — GROUPING BY SPEAKER",
    downloader.group_by_author
)


# ============================================================
# STEP 3/6 — ANALYZING CORE / UNKNOWN
# ============================================================
# >= 1000 speakers -> CORE
# < 1000 speakers  -> UNKNOWN

run_step(
    "STEP 3/6 — ANALYZING CORE / UNKNOWN",
    downloader.analyze_data
)


# ------------------------------------------------------------
# SAVE CORE / UNKNOWN WORD LIST BEFORE BALANCING
# ------------------------------------------------------------
REPORT_DIR = Path(
    "/kaggle/working/reports/fs_gsc_pipeline"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for filename in [
    "core_words.txt",
    "other_words.txt",
]:

    src = root / filename

    if src.exists():

        dst = REPORT_DIR / filename

        dst.write_text(
            src.read_text()
        )

        print("Saved:", dst)


# ============================================================
# STEP 4/6 — BALANCING DATASET
# ============================================================

run_step(
    "STEP 4/6 — BALANCING DATASET",
    downloader.uniform_data
)


# ============================================================
# STEP 5/6 — GENERATING SPLITS
# ============================================================

run_step(
    "STEP 5/6 — GENERATING SPLITS",
    downloader.generate_splits
)


# ------------------------------------------------------------
# SAVE GENERATED SPLITS BEFORE CLEANUP
# ------------------------------------------------------------
for relative_path in [

    "core/train.txt",
    "core/val.txt",
    "core/test.txt",

    "_unknown_/train.txt",
    "_unknown_/val.txt",
    "_unknown_/test.txt",
]:

    src = root / relative_path

    if src.exists():

        safe_name = relative_path.replace("/", "_")

        dst = REPORT_DIR / safe_name

        dst.write_text(
            src.read_text()
        )

        print("Saved:", dst)


# ============================================================
# STEP 6/6 — CLEANUP
# ============================================================

run_step(
    "STEP 6/6 — CLEANUP",
    downloader.remove_unwanted_files
)


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("FS-GSC PREPARATION COMPLETE")
print("=" * 80)

print("\nDataset root:")
print(root)

print("\nTop-level entries:")

for p in sorted(root.iterdir()):
    print("-", p.name)


print("\nSaved report files:")

for p in sorted(REPORT_DIR.iterdir()):
    print("-", p.name)

/kaggle/working/Few-Shot-KWS
Paper script exists: True
✅ Paper module loaded successfully.
✅ Patched old torchaudio.load() API

Dataset path:
/kaggle/working/Few-Shot-KWS/speech_commands

Keyword folders:
36
Direct WAV files inside keyword folders: 105829

STEP 1/6 — FILTERING (<1s REMOVED)
Scanning files..
10435 samples with audio < 1 second removed
95394 samples retained

✅ Finished: STEP 1/6 — FILTERING (<1s REMOVED)
Elapsed: 6.15 min

STEP 2/6 — GROUPING BY SPEAKER
Grouping audio samples by speakers ..

✅ Finished: STEP 2/6 — GROUPING BY SPEAKER
Elapsed: 0.06 min

STEP 3/6 — ANALYZING CORE / UNKNOWN


NotADirectoryError: [Errno 20] Not a directory: '/kaggle/working/Few-Shot-KWS/speech_commands/core/train.txt'

In [40]:
# ============================================================
# CELL 10 — FINAL FS-GSC AUDIT AFTER PAPER PREPROCESSING
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import soundfile as sf

ROOT = Path(
    "/kaggle/working/Few-Shot-KWS/speech_commands"
)

CORE_ROOT = ROOT / "core"
UNKNOWN_ROOT = ROOT / "_unknown_"

REPORT_DIR = Path(
    "/kaggle/working/reports/fs_gsc_final_audit"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 80)
print("FINAL FS-GSC AUDIT")
print("=" * 80)


# ------------------------------------------------------------
# 1. CORE CLASSES
# ------------------------------------------------------------
core_class_dirs = sorted(
    p for p in CORE_ROOT.iterdir()
    if p.is_dir()
)

core_classes = [
    p.name
    for p in core_class_dirs
]

print("\nCORE")
print("-" * 80)

print("Core classes:", len(core_classes))


# ------------------------------------------------------------
# 2. COUNT CORE WAV FILES PER CLASS
# ------------------------------------------------------------
core_counts = {}

core_wav_files = []

for class_dir in core_class_dirs:

    files = list(
        class_dir.glob("*.wav")
    )

    core_counts[class_dir.name] = len(files)

    core_wav_files.extend(files)


core_count_df = pd.DataFrame(
    sorted(
        core_counts.items()
    ),
    columns=[
        "class",
        "num_samples"
    ]
)

print(
    "Total core WAV:",
    len(core_wav_files)
)

print("\nCore samples/class:")
print(
    core_count_df
    .to_string(index=False)
)

print("\nCore sample count stats:")
print(
    core_count_df["num_samples"]
    .describe()
)


# ------------------------------------------------------------
# 3. UNKNOWN DATASET
# ------------------------------------------------------------
unknown_class_dirs = sorted(
    p for p in UNKNOWN_ROOT.iterdir()
    if p.is_dir()
)

unknown_classes = [
    p.name
    for p in unknown_class_dirs
]

unknown_counts = {}

unknown_wav_files = []

for class_dir in unknown_class_dirs:

    files = list(
        class_dir.glob("*.wav")
    )

    unknown_counts[
        class_dir.name
    ] = len(files)

    unknown_wav_files.extend(
        files
    )


unknown_count_df = pd.DataFrame(
    sorted(
        unknown_counts.items()
    ),
    columns=[
        "class",
        "num_samples"
    ]
)

print("\n" + "=" * 80)
print("UNKNOWN")
print("=" * 80)

print(
    "Unknown classes:",
    len(unknown_classes)
)

print(
    "Total unknown WAV:",
    len(unknown_wav_files)
)

print("\nUnknown samples/class:")
print(
    unknown_count_df
    .to_string(index=False)
)


# ------------------------------------------------------------
# 4. READ CORE SPLITS
# ------------------------------------------------------------
def read_lines(path):
    if not path.exists():
        return []

    return [
        line.strip()
        for line in path.read_text().splitlines()
        if line.strip()
    ]


core_train = read_lines(
    CORE_ROOT / "train.txt"
)

core_val = read_lines(
    CORE_ROOT / "val.txt"
)

core_test = read_lines(
    CORE_ROOT / "test.txt"
)


print("\n" + "=" * 80)
print("CORE SPLITS")
print("=" * 80)

print(
    "Train classes:",
    len(core_train)
)

print(
    "Validation classes:",
    len(core_val)
)

print(
    "Test classes:",
    len(core_test)
)

print("\nTrain:")
print(core_train)

print("\nValidation:")
print(core_val)

print("\nTest:")
print(core_test)


# ------------------------------------------------------------
# 5. READ UNKNOWN SPLITS
# ------------------------------------------------------------
unknown_train = read_lines(
    UNKNOWN_ROOT / "train.txt"
)

unknown_val = read_lines(
    UNKNOWN_ROOT / "val.txt"
)

unknown_test = read_lines(
    UNKNOWN_ROOT / "test.txt"
)


print("\n" + "=" * 80)
print("UNKNOWN SPLITS")
print("=" * 80)

print(
    "Unknown train samples:",
    len(unknown_train)
)

print(
    "Unknown validation samples:",
    len(unknown_val)
)

print(
    "Unknown test samples:",
    len(unknown_test)
)

print(
    "Unknown total split entries:",
    len(unknown_train)
    + len(unknown_val)
    + len(unknown_test)
)


# ------------------------------------------------------------
# 6. AUDIO METADATA SAMPLE
# ------------------------------------------------------------
sample_files = (
    core_wav_files[:10]
    +
    unknown_wav_files[:10]
)

metadata_rows = []

for wav_path in sample_files:

    info = sf.info(
        wav_path
    )

    metadata_rows.append({
        "path":
            str(wav_path),

        "sample_rate":
            info.samplerate,

        "channels":
            info.channels,

        "frames":
            info.frames,

        "duration_sec":
            info.frames /
            info.samplerate,

        "format":
            info.format,

        "subtype":
            info.subtype
    })


metadata_df = pd.DataFrame(
    metadata_rows
)

print("\n" + "=" * 80)
print("AUDIO METADATA SAMPLE")
print("=" * 80)

print("\nSample rate:")
print(
    metadata_df[
        "sample_rate"
    ].value_counts()
)

print("\nChannels:")
print(
    metadata_df[
        "channels"
    ].value_counts()
)

print("\nFormat:")
print(
    metadata_df[
        "format"
    ].value_counts()
)

print("\nSubtype:")
print(
    metadata_df[
        "subtype"
    ].value_counts()
)

print("\nDuration:")
print(
    metadata_df[
        "duration_sec"
    ].describe()
)


# ------------------------------------------------------------
# 7. RAW VS PREPARED SUMMARY
# ------------------------------------------------------------
# Raw audit values already measured earlier.
# These are real observed values from this Kaggle session/project.

RAW_TOTAL_WAV = 105835
RAW_KEYWORD_CLASSES = 35
RAW_SHORTER_THAN_1S = 10435

prepared_keyword_wav = (
    len(core_wav_files)
    +
    len(unknown_wav_files)
)

summary = pd.DataFrame([
    {
        "metric":
            "Total raw WAV",

        "raw":
            RAW_TOTAL_WAV,

        "prepared":
            None
    },

    {
        "metric":
            "Keyword classes",

        "raw":
            RAW_KEYWORD_CLASSES,

        "prepared":
            len(core_classes)
            + len(unknown_classes)
    },

    {
        "metric":
            "Files < 1 sec removed",

        "raw":
            RAW_SHORTER_THAN_1S,

        "prepared":
            0
    },

    {
        "metric":
            "Core classes",

        "raw":
            None,

        "prepared":
            len(core_classes)
    },

    {
        "metric":
            "Unknown classes",

        "raw":
            None,

        "prepared":
            len(unknown_classes)
    },

    {
        "metric":
            "Prepared keyword WAV",

        "raw":
            None,

        "prepared":
            prepared_keyword_wav
    }
])


print("\n" + "=" * 80)
print("RAW VS PREPARED SUMMARY")
print("=" * 80)

print(
    summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 8. SAVE REPORTS
# ------------------------------------------------------------
core_count_df.to_csv(
    REPORT_DIR /
    "core_class_distribution.csv",
    index=False
)

unknown_count_df.to_csv(
    REPORT_DIR /
    "unknown_class_distribution.csv",
    index=False
)

metadata_df.to_csv(
    REPORT_DIR /
    "audio_metadata_sample.csv",
    index=False
)

summary.to_csv(
    REPORT_DIR /
    "raw_vs_prepared_summary.csv",
    index=False
)

pd.DataFrame({
    "split":
        ["train", "val", "test"],

    "num_classes":
        [
            len(core_train),
            len(core_val),
            len(core_test)
        ]
}).to_csv(
    REPORT_DIR /
    "core_split_summary.csv",
    index=False
)

pd.DataFrame({
    "split":
        ["train", "val", "test"],

    "num_samples":
        [
            len(unknown_train),
            len(unknown_val),
            len(unknown_test)
        ]
}).to_csv(
    REPORT_DIR /
    "unknown_split_summary.csv",
    index=False
)


print("\n" + "=" * 80)
print("REPORT SAVED")
print("=" * 80)

print(REPORT_DIR)

for p in sorted(
    REPORT_DIR.iterdir()
):
    print("-", p.name)

FINAL FS-GSC AUDIT

CORE
--------------------------------------------------------------------------------
Core classes: 30
Total core WAV: 31860

Core samples/class:
 class  num_samples
   bed         1062
  bird         1062
   cat         1062
   dog         1062
  down         1062
 eight         1062
  five         1062
  four         1062
    go         1062
 happy         1062
 house         1062
  left         1062
marvin         1062
  nine         1062
    no         1062
   off         1062
    on         1062
   one         1062
 right         1062
 seven         1062
sheila         1062
   six         1062
  stop         1062
 three         1062
  tree         1062
   two         1062
    up         1062
   wow         1062
   yes         1062
  zero         1062

Core sample count stats:
count      30.0
mean     1062.0
std         0.0
min      1062.0
25%      1062.0
50%      1062.0
75%      1062.0
max      1062.0
Name: num_samples, dtype: float64

UNKNOWN
Unknown classes: 

In [41]:
# ============================================================
# CELL 11 — INSPECT FEATURE EXTRACTION PIPELINE
# ============================================================

from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

files_to_check = [
    REPO / "protonets/data/FewShotSpeechData.py",
    REPO / "protonets/utils/data.py",
    REPO / "scripts/train/few_shot/train.py",
    REPO / "scripts/train/few_shot/trainval.py",
]

keywords = [
    "mfcc",
    "mel",
    "spectrogram",
    "torchaudio",
    "transform",
    "feature",
    "sample_rate",
    "n_mfcc",
    "n_fft",
    "hop",
    "window",
    "normalize",
]

for file_path in files_to_check:

    print("\n" + "=" * 100)
    print("FILE:", file_path.relative_to(REPO))
    print("=" * 100)

    if not file_path.exists():
        print("NOT FOUND")
        continue

    lines = file_path.read_text(
        errors="ignore"
    ).splitlines()

    matches = []

    for i, line in enumerate(lines):

        if any(
            keyword.lower() in line.lower()
            for keyword in keywords
        ):
            matches.append(i)

    # show context around matching lines
    shown = set()

    for i in matches:

        start = max(0, i - 4)
        end = min(len(lines), i + 6)

        for j in range(start, end):

            if j in shown:
                continue

            print(
                f"{j + 1:4d}: {lines[j]}"
            )

            shown.add(j)

        print("-" * 80)


FILE: protonets/data/FewShotSpeechData.py
   4: import numpy as np
   5: import glob
   6: 
   7: from torch.utils.data import Dataset
   8: from torchvision import transforms
   9: from torchnet.transform import compose
  10: from torchnet.dataset import ListDataset, TransformDataset
  11: import torchaudio
  12: import torch.nn.functional as F
  13: from protonets.data.base import convert_dict, CudaTransform
--------------------------------------------------------------------------------
  14: 
--------------------------------------------------------------------------------
  15: 
--------------------------------------------------------------------------------
  16: class FewShotSpeechDataset(TransformDataset):
--------------------------------------------------------------------------------
  17: 
  18:     def __init__(self, data_dir, class_file, n_support, n_query, cuda, args):
--------------------------------------------------------------------------------
  19:         self.samp

In [42]:
# ============================================================
# CELL 11B — FIND TD-RESNET MODEL + INPUT SHAPE
# ============================================================

from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

search_terms = [
    "TDResNet",
    "TD_ResNet",
    "tdresnet",
    "mfcc",
    "input_size",
    "embedding",
]

for py_file in REPO.rglob("*.py"):

    text = py_file.read_text(
        errors="ignore"
    )

    if any(
        term.lower() in text.lower()
        for term in search_terms
    ):

        print("\n" + "=" * 100)
        print(py_file.relative_to(REPO))
        print("=" * 100)

        lines = text.splitlines()

        for i, line in enumerate(lines):

            if any(
                term.lower() in line.lower()
                for term in search_terms
            ):
                start = max(0, i - 3)
                end = min(len(lines), i + 5)

                for j in range(start, end):
                    print(
                        f"{j + 1:4d}: {lines[j]}"
                    )

                print("-" * 60)


protonets/data/FewShotSpeechData.py
  36:         self.n_support = n_support
  37:         self.n_query = n_query
  38:         self.background_data = self.load_background_data()
  39:         self.mfcc = self.build_mfcc_extractor()
  40:         self.transforms = [partial(convert_dict, 'class'), 
  41:                            self.load_class_samples,
  42:                            self.extract_episode]
  43:         if cuda:
------------------------------------------------------------
  55:                 background_data.append(bg_sound.flatten())
  56:         return background_data
  57:     
  58:     def build_mfcc_extractor(self):
  59:         frame_len = self.window_size_ms / 1000
  60:         stride = self.window_stride_ms / 1000
  61:         mfcc = torchaudio.transforms.MFCC(self.sample_rate,
  62:                                         n_mfcc=self.feature_bin_count,
------------------------------------------------------------
  58:     def build_mfcc_extractor(self

In [43]:
# CELL 12 — VERIFY REAL MFCC SHAPE

import torch
import torchaudio

sample_rate = 16000
window_ms = 40
stride_ms = 20
n_mfcc = 40

mfcc = torchaudio.transforms.MFCC(
    sample_rate=sample_rate,
    n_mfcc=n_mfcc,
    melkwargs={
        "hop_length": int(0.020 * sample_rate),
        "n_fft": int(0.040 * sample_rate),
    }
)

waveform = torch.zeros(1, 16000)

features = mfcc(waveform)

print("MFCC raw shape:", features.shape)

features = features[0].T
print("After [0].T:", features.shape)

features = torch.unsqueeze(features, 0)
print("Final model input shape:", features.shape)


MFCC raw shape: torch.Size([1, 40, 51])
After [0].T: torch.Size([51, 40])
Final model input shape: torch.Size([1, 51, 40])


In [44]:
# ============================================================
# CELL 13 — REPORT 2.5 VISUALIZE RAW VS PROCESSED FEATURES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import torch
import torchaudio

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
FIG_DIR = Path("/kaggle/working/reports/figures/2_5")
FIG_DIR.mkdir(parents=True, exist_ok=True)

GSC_ROOT = Path("/kaggle/working/Few-Shot-KWS/speech_commands")

AUDIOMNIST_RAW = Path(
    "/kaggle/input/datasets/sripaadsrinivasan/audio-mnist/data"
)

AUDIOMNIST_PROCESSED = Path(
    "/kaggle/working/processed/audiomnist_16k_1s"
)

FSC_ROOT = Path(
    "/kaggle/input/datasets/muhammadnaseem2020/"
    "fluent-speech-commands/fluent_speech_commands_dataset"
)

FSC_PROCESSED = Path(
    "/kaggle/working/processed/fsc_segments_16k_1s"
)

TARGET_SR = 16000


# ------------------------------------------------------------
# FEATURE EXTRACTORS — SAME MFCC CONFIG AS PAPER
# ------------------------------------------------------------
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=TARGET_SR,
    n_mfcc=40,
    melkwargs={
        "hop_length": 320,   # 20 ms
        "n_fft": 640,        # 40 ms
    }
)

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=640,
    hop_length=320,
    n_mels=40
)

spec_transform = torchaudio.transforms.Spectrogram(
    n_fft=640,
    hop_length=320,
    power=2.0
)


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def load_audio(path):
    audio, sr = sf.read(path, dtype="float32")

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    return audio, sr


def to_tensor(audio):
    return torch.tensor(audio, dtype=torch.float32).unsqueeze(0)


def save_waveform(audio, sr, title, filename):
    t = np.arange(len(audio)) / sr

    plt.figure(figsize=(10, 3))
    plt.plot(t, audio)
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=200)
    plt.close()


def save_spectrogram(audio, sr, title, filename):
    x = to_tensor(audio)

    spec = spec_transform(x)[0].numpy()

    spec_db = 10 * np.log10(spec + 1e-10)

    plt.figure(figsize=(10, 4))
    plt.imshow(
        spec_db,
        origin="lower",
        aspect="auto"
    )
    plt.title(title)
    plt.xlabel("Time frame")
    plt.ylabel("Frequency bin")
    plt.colorbar(label="dB")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=200)
    plt.close()


def save_mel(audio, sr, title, filename):
    x = to_tensor(audio)

    mel = mel_transform(x)[0].numpy()
    mel_db = 10 * np.log10(mel + 1e-10)

    plt.figure(figsize=(10, 4))
    plt.imshow(
        mel_db,
        origin="lower",
        aspect="auto"
    )
    plt.title(title)
    plt.xlabel("Time frame")
    plt.ylabel("Mel bin")
    plt.colorbar(label="dB")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=200)
    plt.close()


def save_mfcc(audio, sr, title, filename):
    x = to_tensor(audio)

    mfcc = mfcc_transform(x)[0].numpy()

    plt.figure(figsize=(10, 4))
    plt.imshow(
        mfcc,
        origin="lower",
        aspect="auto"
    )
    plt.title(title)
    plt.xlabel("Time frame")
    plt.ylabel("MFCC coefficient")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=200)
    plt.close()


def rms(audio):
    return float(np.sqrt(np.mean(audio ** 2)))


def peak(audio):
    return float(np.max(np.abs(audio)))


def summarize_audio(name, audio, sr):
    return {
        "name": name,
        "sample_rate": sr,
        "samples": len(audio),
        "duration_sec": len(audio) / sr,
        "peak_amplitude": peak(audio),
        "rms": rms(audio),
    }


def make_all_figures(prefix, audio, sr, title):
    save_waveform(
        audio, sr,
        f"{title} — Waveform",
        f"{prefix}_waveform.png"
    )

    save_spectrogram(
        audio, sr,
        f"{title} — Spectrogram",
        f"{prefix}_spectrogram.png"
    )

    # Mel/MFCC require 16k input in this report pipeline
    if sr == TARGET_SR:
        save_mel(
            audio, sr,
            f"{title} — Mel-spectrogram",
            f"{prefix}_mel.png"
        )

        save_mfcc(
            audio, sr,
            f"{title} — MFCC",
            f"{prefix}_mfcc.png"
        )


# ============================================================
# 1. FS-GSC — PROCESSED SAMPLE
# ============================================================

gsc_candidates = list((GSC_ROOT / "core").rglob("*.wav"))

if len(gsc_candidates) == 0:
    raise RuntimeError("Không tìm thấy processed FS-GSC WAV.")

gsc_path = gsc_candidates[0]

gsc_audio, gsc_sr = load_audio(gsc_path)

make_all_figures(
    "gsc_processed",
    gsc_audio,
    gsc_sr,
    f"FS-GSC processed — {gsc_path.parent.name}"
)

rows = [
    summarize_audio(
        "FS-GSC processed",
        gsc_audio,
        gsc_sr
    )
]


# ============================================================
# 2. AUDIOMNIST — RAW VS PROCESSED SAME FILE
# ============================================================

am_raw_candidates = list(AUDIOMNIST_RAW.glob("*/*.wav"))

if len(am_raw_candidates) == 0:
    raise RuntimeError("Không tìm thấy AudioMNIST raw WAV.")

am_raw_path = am_raw_candidates[0]

speaker = am_raw_path.parent.name

am_processed_path = (
    AUDIOMNIST_PROCESSED /
    speaker /
    am_raw_path.name
)

am_raw_audio, am_raw_sr = load_audio(am_raw_path)
am_proc_audio, am_proc_sr = load_audio(am_processed_path)

make_all_figures(
    "audiomnist_raw",
    am_raw_audio,
    am_raw_sr,
    f"AudioMNIST raw — {am_raw_path.name}"
)

make_all_figures(
    "audiomnist_processed",
    am_proc_audio,
    am_proc_sr,
    f"AudioMNIST processed — {am_processed_path.name}"
)

rows.append(
    summarize_audio(
        "AudioMNIST raw",
        am_raw_audio,
        am_raw_sr
    )
)

rows.append(
    summarize_audio(
        "AudioMNIST processed",
        am_proc_audio,
        am_proc_sr
    )
)


# ============================================================
# 3. FSC — RAW UTTERANCE VS FIRST PROCESSED SEGMENT
# ============================================================

fsc_meta = pd.read_csv(
    FSC_ROOT / "data" / "train_data.csv"
)

fsc_row = fsc_meta.iloc[0]

fsc_raw_path = FSC_ROOT / fsc_row["path"]

utterance_id = Path(fsc_row["path"]).stem

segment_candidates = sorted(
    FSC_PROCESSED.rglob(
        f"{utterance_id}/segment_*.wav"
    )
)

if len(segment_candidates) == 0:
    raise RuntimeError(
        f"Không tìm thấy segment processed cho {utterance_id}"
    )

fsc_segment_path = segment_candidates[0]

fsc_raw_audio, fsc_raw_sr = load_audio(fsc_raw_path)
fsc_seg_audio, fsc_seg_sr = load_audio(fsc_segment_path)

make_all_figures(
    "fsc_raw",
    fsc_raw_audio,
    fsc_raw_sr,
    f"FSC raw — {fsc_row['transcription']}"
)

make_all_figures(
    "fsc_segment",
    fsc_seg_audio,
    fsc_seg_sr,
    f"FSC processed segment — {fsc_row['transcription']}"
)

rows.append(
    summarize_audio(
        "FSC raw utterance",
        fsc_raw_audio,
        fsc_raw_sr
    )
)

rows.append(
    summarize_audio(
        "FSC processed segment",
        fsc_seg_audio,
        fsc_seg_sr
    )
)


# ============================================================
# 4. SAVE SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame(rows)

summary_df.to_csv(
    FIG_DIR / "audio_comparison_summary.csv",
    index=False
)

print("=" * 80)
print("REPORT 2.5 VISUALIZATION COMPLETE")
print("=" * 80)

print(summary_df.to_string(index=False))

print("\nSaved figures:")

for p in sorted(FIG_DIR.iterdir()):
    print("-", p.name)

REPORT 2.5 VISUALIZATION COMPLETE
                 name  sample_rate  samples  duration_sec  peak_amplitude      rms
     FS-GSC processed        16000    16000      1.000000        0.637634 0.043546
       AudioMNIST raw        48000    39797      0.829104        0.016907 0.002114
 AudioMNIST processed        16000    16000      1.000000        1.000000 0.115548
    FSC raw utterance        16000    29722      1.857625        0.138458 0.018968
FSC processed segment        16000    16000      1.000000        0.999969 0.165175

Saved figures:
- audio_comparison_summary.csv
- audiomnist_processed_mel.png
- audiomnist_processed_mfcc.png
- audiomnist_processed_spectrogram.png
- audiomnist_processed_waveform.png
- audiomnist_raw_spectrogram.png
- audiomnist_raw_waveform.png
- fsc_raw_mel.png
- fsc_raw_mfcc.png
- fsc_raw_spectrogram.png
- fsc_raw_waveform.png
- fsc_segment_mel.png
- fsc_segment_mfcc.png
- fsc_segment_spectrogram.png
- fsc_segment_waveform.png
- gsc_processed_mel.png
- gsc_pr

In [45]:
# ============================================================
# DOWNLOAD REPORT 2.5 FIGURES + CSV AS ZIP
# ============================================================

from pathlib import Path
import shutil

SRC_DIR = Path("/kaggle/working/reports/figures/2_5")
ZIP_BASE = Path("/kaggle/working/report_2_5_figures")

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục: {SRC_DIR}")

zip_path = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(SRC_DIR)
)

print("✅ ZIP created:")
print(zip_path)

print("\nFiles included:")
for p in sorted(SRC_DIR.iterdir()):
    print("-", p.name)

✅ ZIP created:
/kaggle/working/report_2_5_figures.zip

Files included:
- audio_comparison_summary.csv
- audiomnist_processed_mel.png
- audiomnist_processed_mfcc.png
- audiomnist_processed_spectrogram.png
- audiomnist_processed_waveform.png
- audiomnist_raw_spectrogram.png
- audiomnist_raw_waveform.png
- fsc_raw_mel.png
- fsc_raw_mfcc.png
- fsc_raw_spectrogram.png
- fsc_raw_waveform.png
- fsc_segment_mel.png
- fsc_segment_mfcc.png
- fsc_segment_spectrogram.png
- fsc_segment_waveform.png
- gsc_processed_mel.png
- gsc_processed_mfcc.png
- gsc_processed_spectrogram.png
- gsc_processed_waveform.png


In [46]:
# ============================================================
# DOWNLOAD ALL REPORT ARTIFACTS AS ZIP
# ============================================================

from pathlib import Path
import shutil

REPORT_ROOT = Path("/kaggle/working/reports")
ZIP_BASE = Path("/kaggle/working/full_report_artifacts")

if not REPORT_ROOT.exists():
    raise FileNotFoundError(f"Không tìm thấy: {REPORT_ROOT}")

zip_path = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(REPORT_ROOT)
)

print("✅ Created:")
print(zip_path)

✅ Created:
/kaggle/working/full_report_artifacts.zip


In [47]:
# ============================================================
# CELL 14 — INSPECT TD-RESNET + PROTONET ARCHITECTURE
# ============================================================

from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

search_terms = [
    "TDResNet",
    "TD_ResNet",
    "tdresnet",
    "ProtoNet",
    "prototypical",
    "prototype",
    "euclidean",
    "distance",
    "embedding",
    "forward",
    "loss",
]

matched_files = []

for py_file in REPO.rglob("*.py"):
    text = py_file.read_text(errors="ignore")

    if any(term.lower() in text.lower() for term in search_terms):
        matched_files.append(py_file)

print("=" * 100)
print("POSSIBLE MODEL / PROTONET FILES")
print("=" * 100)

for p in matched_files:
    print(p.relative_to(REPO))


print("\n" + "=" * 100)
print("RELEVANT CODE SNIPPETS")
print("=" * 100)

for py_file in matched_files:

    text = py_file.read_text(errors="ignore")
    lines = text.splitlines()

    found = False

    for i, line in enumerate(lines):

        if any(term.lower() in line.lower() for term in search_terms):

            if not found:
                print("\n" + "=" * 100)
                print("FILE:", py_file.relative_to(REPO))
                print("=" * 100)
                found = True

            start = max(0, i - 5)
            end = min(len(lines), i + 10)

            for j in range(start, end):
                print(f"{j+1:4d}: {lines[j]}")

            print("-" * 80)

POSSIBLE MODEL / PROTONET FILES
setup.py
protonets/engine.py
scripts/plot.py
scripts/parse_results.py
protonets/utils/model.py
protonets/utils/data.py
protonets/models/few_shot.py
protonets/models/__init__.py
protonets/models/utils.py
protonets/data/FewShotSpeechData.py
protonets/models/encoder/GoogleKWS.py
protonets/models/encoder/default.py
protonets/models/encoder/TCResNet.py
protonets/models/encoder/baseUtil.py
scripts/predict/few_shot/run_eval.py
scripts/predict/few_shot/eval.py
scripts/train/few_shot/run_train.py
scripts/train/few_shot/train.py
scripts/train/few_shot/run_trainval.py
scripts/train/few_shot/trainval.py

RELEVANT CODE SNIPPETS

FILE: setup.py
   1: from setuptools import setup
   2: 
   3: setup(name='protonets',
   4:       version='0.0.1',
   5:       author='Jake Snell',
   6:       author_email='jsnell@cs.toronto.edu',
   7:       license='MIT',
   8:       packages=['protonets'],
   9:       install_requires=[
  10:           'torch',
  11:           'tqdm'
  1

In [48]:
# ============================================================
# CELL 14B — FIND MODEL CONFIG / EMBEDDING DIMENSION
# ============================================================

from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

terms = [
    "embedding_dim",
    "output_dim",
    "hidden",
    "channels",
    "num_blocks",
    "resnet",
    "model",
    "distance",
    "euclidean",
]

for py_file in REPO.rglob("*.py"):

    lines = py_file.read_text(errors="ignore").splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(t.lower() in line.lower() for t in terms)
    ]

    if not matches:
        continue

    print("\n" + "=" * 100)
    print(py_file.relative_to(REPO))
    print("=" * 100)

    shown = set()

    for i in matches:
        start = max(0, i - 3)
        end = min(len(lines), i + 7)

        for j in range(start, end):
            if j not in shown:
                print(f"{j+1:4d}: {lines[j]}")
                shown.add(j)

        print("-" * 60)


protonets/engine.py
  11: 
  12:     def train(self, **kwargs):
  13:         state = {
  14:             'model': kwargs['model'],
  15:             'loader': kwargs['loader'],
  16:             'optim_method': kwargs['optim_method'],
  17:             'optim_config': kwargs['optim_config'],
  18:             'max_epoch': kwargs['max_epoch'],
  19:             'epoch': 0, # epochs done so far
  20:             't': 0, # samples seen so far
------------------------------------------------------------
  22:             'stop': False
  23:         }
  24: 
  25:         state['optimizer'] = state['optim_method'](state['model'].parameters(), **state['optim_config'])
  26: 
  27:         self.hooks['on_start'](state)
  28:         while state['epoch'] < state['max_epoch'] and not state['stop']:
  29:             state['model'].train()
  30: 
  31:             self.hooks['on_start_epoch'](state)
------------------------------------------------------------
  32: 
  33:             state['ep

In [49]:
# ============================================================
# CELL 15 — ENVIRONMENT / DEPENDENCY CHECK
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import sys
import torch
import torchaudio
import torchvision
import numpy as np

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchaudio  :", torchaudio.__version__)
print("Torchvision :", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA capability:",
        torch.cuda.get_device_capability(0)
    )

print("\nRepository:")
!pwd

print("\nrequirements.txt:")
!cat requirements.txt

/kaggle/working/Few-Shot-KWS
ENVIRONMENT CHECK
Python      : 3.12.13
PyTorch     : 2.10.0+cu128
Torchaudio  : 2.10.0+cu128
Torchvision : 0.25.0+cu128

CUDA available: True
GPU: Tesla T4
CUDA capability: (7, 5)

Repository:
/kaggle/working/Few-Shot-KWS

requirements.txt:
torch
torchnet
torchaudio
torchvision

In [6]:
# ============================================================
# INSTALL MISSING TORCHNET DEPENDENCY
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "torchnet"
])

print("✅ torchnet installed")

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.3 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 50.5 MB/s eta 0:00:00
  Created wheel for torchnet: filename=torchnet-0.0.4-py3-none-any.whl size=29770 sha256=ff55393011d41005c195f6420d38f4bf15db6ae5a75dcf308b683627824df106
  Stored in directory: /root/.cache/pip/wheels/69/12/36/73a61a7ea788c74f258d51c041265c2a5c06dec6ec04ca847b
  Created wheel for visdom: filename=visdom-0.3.0-py3-none-any.whl size=2014861 sha256=2eda44a0a2132ebb34b9e9c94c5b624330abd9a4c9de6c2f8d16174dae61092f

In [7]:
# ============================================================
# HARD FIX — FORCE LOAD LOCAL protonets PACKAGE
# ============================================================

import sys
import os
import types
import importlib
import importlib.util
from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")
PKG  = REPO / "protonets"

print("Repo exists       :", REPO.exists())
print("Package exists    :", PKG.exists())
print("__init__.py exists:", (PKG / "__init__.py").exists())

print("\nPython executable:")
print(sys.executable)

# ------------------------------------------------------------
# 1. Ensure repository root is visible
# ------------------------------------------------------------
repo_str = str(REPO)

if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

importlib.invalidate_caches()

print("\nRepo in sys.path:", repo_str in sys.path)

# ------------------------------------------------------------
# 2. Check whether Python can discover protonets normally
# ------------------------------------------------------------
spec = importlib.util.find_spec("protonets")

print("find_spec('protonets'):", spec)

# ------------------------------------------------------------
# 3. If Kaggle still cannot discover it, register the local
#    directory manually as a Python package.
# ------------------------------------------------------------
if spec is None:
    print("\n⚠️ Normal import discovery failed.")
    print("Registering local protonets package manually...")

    package = types.ModuleType("protonets")

    package.__file__ = str(PKG / "__init__.py")
    package.__path__ = [str(PKG)]
    package.__package__ = "protonets"

    sys.modules["protonets"] = package

    # Execute protonets/__init__.py inside the package namespace
    init_code = (PKG / "__init__.py").read_text()

    exec(
        compile(
            init_code,
            str(PKG / "__init__.py"),
            "exec"
        ),
        package.__dict__
    )

# ------------------------------------------------------------
# 4. Test package
# ------------------------------------------------------------
import protonets

print("\n✅ protonets loaded")
print("Package path:", protonets.__path__)

# ------------------------------------------------------------
# 5. Test important submodules
# ------------------------------------------------------------
from protonets.models.encoder.TCResNet import TCResNet8
from protonets.models.few_shot import Protonet
from protonets.data.FewShotSpeechData import FewShotSpeechDataset

print("✅ TCResNet8 import OK")
print("✅ Protonet import OK")
print("✅ FewShotSpeechDataset import OK")

Repo exists       : True
Package exists    : True
__init__.py exists: True

Python executable:
/usr/bin/python3

Repo in sys.path: True
find_spec('protonets'): ModuleSpec(name='protonets', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bf758f64470>, origin='/kaggle/working/Few-Shot-KWS/protonets/__init__.py', submodule_search_locations=['/kaggle/working/Few-Shot-KWS/protonets'])

✅ protonets loaded
Package path: ['/kaggle/working/Few-Shot-KWS/protonets']
✅ TCResNet8 import OK
✅ Protonet import OK
✅ FewShotSpeechDataset import OK


In [62]:
# ============================================================
# CELL 16 — PATCH REPO FOR CURRENT TORCHAUDIO
# ============================================================

import torchaudio
from protonets.data.FewShotSpeechData import FewShotSpeechDataset

def load_audio_fixed(self, key, out_field, d):
    sound, _ = torchaudio.load(
        d[key],
        num_frames=self.desired_samples
    )

    d[out_field] = sound

    return d


FewShotSpeechDataset.load_audio = load_audio_fixed

print("✅ FewShotSpeechDataset.load_audio patched successfully")

✅ FewShotSpeechDataset.load_audio patched successfully


In [63]:
# ============================================================
# CELL 17 — TCRESNET8 FORWARD PASS TEST
# ============================================================

import torch
from protonets.models.encoder.TCResNet import TCResNet8

BATCH_SIZE = 8

# MFCC input đã xác nhận ở 2.4:
# [batch, channel, time, mfcc]
x = torch.randn(
    BATCH_SIZE,
    1,
    51,
    40
)

print("=" * 70)
print("TCRESNET8 FORWARD TEST")
print("=" * 70)

print("Input shape:", x.shape)

# Build encoder
encoder = TCResNet8(
    in_c=1,
    in_h=51,
    in_w=40
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder = encoder.to(device)
x = x.to(device)

print("Device:", device)

# Forward pass
encoder.eval()

with torch.no_grad():
    z = encoder(x)

print("\nOutput embedding shape:", z.shape)
print("Embedding dimension:", z.shape[-1])

# Parameter count
total_params = sum(
    p.numel()
    for p in encoder.parameters()
)

trainable_params = sum(
    p.numel()
    for p in encoder.parameters()
    if p.requires_grad
)

print("\nTotal parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\n✅ TCResNet8 forward pass successful")

TCRESNET8 FORWARD TEST
Input shape: torch.Size([8, 1, 51, 40])
Device: cuda

Output embedding shape: torch.Size([8, 48])
Embedding dimension: 48

Total parameters: 64,560
Trainable parameters: 64,560

✅ TCResNet8 forward pass successful


In [64]:
# ============================================================
# CELL 18 — PROTONET EPISODE TEST
# ============================================================

import torch

from protonets.models.few_shot import Protonet
from protonets.models.encoder.TCResNet import TCResNet8

# ------------------------------------------------------------
# Episode configuration
# ------------------------------------------------------------
N_WAY = 5
N_SHOT = 1
N_QUERY = 3

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# Build encoder + ProtoNet
# ------------------------------------------------------------
encoder = TCResNet8(
    in_c=1,
    in_h=51,
    in_w=40
)

model = Protonet(
    encoder=encoder,
    encoding="TCResNet8"
).to(device)

# ------------------------------------------------------------
# Fake MFCC episode
#
# xs = [way, shot, channel, time, mfcc]
# xq = [way, query, channel, time, mfcc]
# ------------------------------------------------------------
xs = torch.randn(
    N_WAY,
    N_SHOT,
    1,
    51,
    40,
    device=device
)

xq = torch.randn(
    N_WAY,
    N_QUERY,
    1,
    51,
    40,
    device=device
)

sample = {
    "xs": xs,
    "xq": xq
}

print("=" * 70)
print("PROTONET EPISODE TEST")
print("=" * 70)

print("Support shape:", xs.shape)
print("Query shape  :", xq.shape)

# ------------------------------------------------------------
# Run loss
# ------------------------------------------------------------
model.eval()

with torch.no_grad():
    loss, output = model.loss(sample)

print("\nLoss:")
print(loss.item())

print("\nOutput:")
print(output)

print("\n✅ ProtoNet episode executed successfully")

PROTONET EPISODE TEST
Support shape: torch.Size([5, 1, 1, 51, 40])
Query shape  : torch.Size([5, 3, 1, 51, 40])

Loss:
1.6094226837158203

Output:
{'loss': 1.6094226837158203, 'acc': 0.2666666805744171}

✅ ProtoNet episode executed successfully


In [65]:
# ============================================================
# CELL 19 — END-TO-END SMOKE TEST WITH REAL FS-GSC EPISODE
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import torch

from protonets.data.FewShotSpeechData import FewShotSpeechDataset
from protonets.models.few_shot import Protonet
from protonets.models.encoder.TCResNet import TCResNet8


# ------------------------------------------------------------
# 1. PATCH OLD TORCHAUDIO API AGAIN IN CURRENT RUNTIME
# ------------------------------------------------------------
import torchaudio

def load_audio_fixed(self, key, out_field, d):
    sound, _ = torchaudio.load(
        d[key],
        num_frames=self.desired_samples
    )
    d[out_field] = sound
    return d

FewShotSpeechDataset.load_audio = load_audio_fixed

print("✅ torchaudio patch active")


# ------------------------------------------------------------
# 2. REAL FS-GSC PATHS
# ------------------------------------------------------------
DATA_DIR = (
    "/kaggle/working/Few-Shot-KWS/"
    "speech_commands/core"
)

CLASS_FILE = (
    "/kaggle/working/Few-Shot-KWS/"
    "speech_commands/core/train.txt"
)


# ------------------------------------------------------------
# 3. SMALL EPISODE FOR SMOKE TEST
# ------------------------------------------------------------
N_WAY = 5
N_SHOT = 1
N_QUERY = 2


# ------------------------------------------------------------
# 4. SPEECH CONFIG — SAME FEATURE PIPELINE AS PAPER
# ------------------------------------------------------------
speech_args = {
    "sample_rate": 16000,
    "clip_duration": 1000,
    "window_size": 40,
    "window_stride": 20,
    "num_features": 40,

    "foreground_volume": 1.0,

    "time_shift": 100,

    # Disable augmentation for first smoke test
    "include_background": False,
    "bg_volume": 0.1,
    "bg_frequency": 1.0,

    "include_silence": False,
    "num_silence": 1000,

    "include_unknown": False,
}


# ------------------------------------------------------------
# 5. BUILD REAL DATASET
# ------------------------------------------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dataset = FewShotSpeechDataset(
    data_dir=DATA_DIR,
    class_file=CLASS_FILE,
    n_support=N_SHOT,
    n_query=N_QUERY,
    cuda=torch.cuda.is_available(),
    args=speech_args
)

print("\nDataset classes:", len(dataset.class_names))
print("First classes:", dataset.class_names[:10])


# ------------------------------------------------------------
# 6. GET FIRST N_WAY CLASSES
#    Each dataset item returns one class episode component.
# ------------------------------------------------------------
episode_items = []

for i in range(N_WAY):
    episode_items.append(dataset[i])


# ------------------------------------------------------------
# 7. STACK SUPPORT + QUERY
# ------------------------------------------------------------
xs = torch.stack(
    [item["xs"] for item in episode_items],
    dim=0
)

xq = torch.stack(
    [item["xq"] for item in episode_items],
    dim=0
)

sample = {
    "xs": xs,
    "xq": xq
}


print("\n" + "=" * 70)
print("REAL EPISODE SHAPES")
print("=" * 70)

print("Support shape:", xs.shape)
print("Query shape  :", xq.shape)

print("\nExpected:")
print(
    f"Support = [{N_WAY}, {N_SHOT}, 1, 51, 40]"
)
print(
    f"Query   = [{N_WAY}, {N_QUERY}, 1, 51, 40]"
)


# ------------------------------------------------------------
# 8. BUILD REAL MODEL
# ------------------------------------------------------------
encoder = TCResNet8(
    in_c=1,
    in_h=51,
    in_w=40
)

model = Protonet(
    encoder=encoder,
    encoding="TCResNet8"
).to(device)


# ------------------------------------------------------------
# 9. RUN REAL EPISODE
# ------------------------------------------------------------
model.eval()

with torch.no_grad():
    loss, output = model.loss(sample)


# ------------------------------------------------------------
# 10. RESULTS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("END-TO-END REAL FS-GSC RESULT")
print("=" * 70)

print("Loss:")
print(loss.item())

print("\nOutput:")
print(output)

print("\n✅ REAL FS-GSC END-TO-END SMOKE TEST SUCCESSFUL")

/kaggle/working/Few-Shot-KWS
✅ torchaudio patch active

Dataset classes: 20
First classes: ['six', 'dog', 'down', 'house', 'yes', 'bed', 'eight', 'bird', 'sheila', 'off']

REAL EPISODE SHAPES
Support shape: torch.Size([5, 1, 1, 51, 40])
Query shape  : torch.Size([5, 2, 1, 51, 40])

Expected:
Support = [5, 1, 1, 51, 40]
Query   = [5, 2, 1, 51, 40]

END-TO-END REAL FS-GSC RESULT
Loss:
1.9865585565567017

Output:
{'loss': 1.9865585565567017, 'acc': 0.20000000298023224}

✅ REAL FS-GSC END-TO-END SMOKE TEST SUCCESSFUL


In [66]:
# ============================================================
# CELL 20 — FS-GSC SMOKE TRAINING
# 5-way 1-shot, real episodic training
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm

from protonets.data.FewShotSpeechData import FewShotSpeechDataset
from protonets.models.encoder.TCResNet import TCResNet8
from protonets.models.few_shot import Protonet


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. PATCH OLD TORCHAUDIO API
# ============================================================

def load_audio_fixed(self, key, out_field, d):
    sound, _ = torchaudio.load(
        d[key],
        num_frames=self.desired_samples
    )

    d[out_field] = sound
    return d


FewShotSpeechDataset.load_audio = load_audio_fixed


# ============================================================
# 3. CONFIG
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_ROOT = Path(
    "/kaggle/working/Few-Shot-KWS/speech_commands/core"
)

TRAIN_FILE = DATA_ROOT / "train.txt"
VAL_FILE   = DATA_ROOT / "val.txt"

# Few-shot setup
N_WAY = 5
N_SHOT = 1
N_QUERY = 5

# Short smoke experiment
EPOCHS = 10

TRAIN_EPISODES = 20
VAL_EPISODES = 10

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0


# ============================================================
# 4. SPEECH FEATURE CONFIG
# ============================================================

speech_args = {
    "sample_rate": 16000,
    "clip_duration": 1000,

    "window_size": 40,
    "window_stride": 20,

    "num_features": 40,

    "foreground_volume": 1.0,

    # paper supports time shifting
    "time_shift": 100,

    # first controlled baseline:
    # disable background/silence/unknown
    "include_background": False,
    "bg_volume": 0.1,
    "bg_frequency": 1.0,

    "include_silence": False,
    "num_silence": 1000,

    "include_unknown": False,
}


print("=" * 70)
print("FS-GSC SMOKE TRAINING")
print("=" * 70)

print("Device          :", DEVICE)
print("N-way           :", N_WAY)
print("N-shot          :", N_SHOT)
print("N-query         :", N_QUERY)
print("Epochs          :", EPOCHS)
print("Train episodes  :", TRAIN_EPISODES)
print("Val episodes    :", VAL_EPISODES)
print("Learning rate   :", LEARNING_RATE)


# ============================================================
# 5. LOAD TRAIN / VALIDATION DATASETS
# ============================================================

train_dataset = FewShotSpeechDataset(
    data_dir=str(DATA_ROOT),
    class_file=str(TRAIN_FILE),

    n_support=N_SHOT,
    n_query=N_QUERY,

    # keep dataset on CPU; move episode to GPU manually
    cuda=False,

    args=speech_args
)

val_dataset = FewShotSpeechDataset(
    data_dir=str(DATA_ROOT),
    class_file=str(VAL_FILE),

    n_support=N_SHOT,
    n_query=N_QUERY,

    cuda=False,

    args=speech_args
)


print("\nTrain classes:", len(train_dataset.class_names))
print(train_dataset.class_names)

print("\nValidation classes:", len(val_dataset.class_names))
print(val_dataset.class_names)


if N_WAY > len(train_dataset.class_names):
    raise ValueError("N_WAY larger than number of train classes")

if N_WAY > len(val_dataset.class_names):
    raise ValueError("N_WAY larger than number of validation classes")


# ============================================================
# 6. EPISODE SAMPLER
# ============================================================

def sample_episode(dataset, n_way):

    class_indices = random.sample(
        range(len(dataset.class_names)),
        n_way
    )

    items = [
        dataset[i]
        for i in class_indices
    ]

    xs = torch.stack(
        [item["xs"] for item in items],
        dim=0
    )

    xq = torch.stack(
        [item["xq"] for item in items],
        dim=0
    )

    return {
        "xs": xs.to(DEVICE),
        "xq": xq.to(DEVICE),
    }


# ============================================================
# 7. MODEL
# ============================================================

encoder = TCResNet8(
    in_c=1,
    in_h=51,
    in_w=40
)

model = Protonet(
    encoder=encoder,
    encoding="TCResNet8"
).to(DEVICE)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


print(
    "\nTrainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)


# ============================================================
# 8. ONE TRAIN / VAL EPOCH
# ============================================================

def run_episodes(
    dataset,
    num_episodes,
    training
):

    if training:
        model.train()
    else:
        model.eval()

    losses = []
    accuracies = []

    iterator = tqdm(
        range(num_episodes),
        leave=False,
        desc="train" if training else "valid"
    )

    for _ in iterator:

        sample = sample_episode(
            dataset,
            N_WAY
        )

        if training:

            optimizer.zero_grad()

            loss, output = model.loss(
                sample
            )

            loss.backward()

            optimizer.step()

        else:

            with torch.no_grad():
                loss, output = model.loss(
                    sample
                )

        losses.append(
            float(output["loss"])
        )

        accuracies.append(
            float(output["acc"])
        )

        iterator.set_postfix(
            loss=f"{np.mean(losses):.4f}",
            acc=f"{np.mean(accuracies):.3f}"
        )

    return (
        float(np.mean(losses)),
        float(np.mean(accuracies))
    )


# ============================================================
# 9. TRAIN
# ============================================================

history = []

best_val_loss = float("inf")

OUTPUT_DIR = Path(
    "/kaggle/working/results/fs_gsc_smoke"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL = (
    OUTPUT_DIR /
    "best_model_state.pt"
)

start_time = time.time()


for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = run_episodes(
        train_dataset,
        TRAIN_EPISODES,
        training=True
    )

    val_loss, val_acc = run_episodes(
        val_dataset,
        VAL_EPISODES,
        training=False
    )

    row = {
        "epoch": epoch,

        "train_loss": train_loss,
        "train_acc": train_acc,

        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss={train_loss:.4f} "
        f"acc={train_acc:.3f} | "
        f"val loss={val_loss:.4f} "
        f"acc={val_acc:.3f}"
    )

    # save best validation checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_loss":
                    val_loss,

                "val_acc":
                    val_acc,

                "config": {
                    "way": N_WAY,
                    "shot": N_SHOT,
                    "query": N_QUERY,
                    "mfcc_shape":
                        [1, 51, 40],
                    "embedding_dim":
                        48,
                }
            },
            BEST_MODEL
        )

        print(
            "   ✅ saved new best model"
        )


elapsed = time.time() - start_time


# ============================================================
# 10. SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(history)

history_df.to_csv(
    OUTPUT_DIR /
    "training_history.csv",
    index=False
)


print("\n" + "=" * 70)
print("SMOKE TRAINING COMPLETE")
print("=" * 70)

print(
    "Elapsed:",
    round(elapsed, 2),
    "seconds"
)

print(
    "Best validation loss:",
    best_val_loss
)

best_row = history_df.loc[
    history_df["val_loss"].idxmin()
]

print("\nBest epoch:")
print(best_row)

print("\nSaved:")
print(BEST_MODEL)
print(
    OUTPUT_DIR /
    "training_history.csv"
)

/kaggle/working/Few-Shot-KWS
FS-GSC SMOKE TRAINING
Device          : cuda
N-way           : 5
N-shot          : 1
N-query         : 5
Epochs          : 10
Train episodes  : 20
Val episodes    : 10
Learning rate   : 0.001

Train classes: 20
['six', 'dog', 'down', 'house', 'yes', 'bed', 'eight', 'bird', 'sheila', 'off', 'four', 'up', 'seven', 'three', 'happy', 'zero', 'right', 'tree', 'go', 'five']

Validation classes: 5
['nine', 'stop', 'on', 'left', 'wow']

Trainable parameters: 64,560


train:   0%|          | 0/20 [00:00<?, ?it/s]

valid:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss=1.6181 acc=0.268 | val loss=1.5832 acc=0.316
   ✅ saved new best model


train:   0%|          | 0/20 [00:00<?, ?it/s]

...

valid:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 09 | train loss=1.2242 acc=0.530 | val loss=1.2788 acc=0.464
   ✅ saved new best model


train:   0%|          | 0/20 [00:00<?, ?it/s]

valid:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 10 | train loss=1.1160 acc=0.502 | val loss=1.2321 acc=0.536
   ✅ saved new best model

SMOKE TRAINING COMPLETE
Elapsed: 117.17 seconds
Best validation loss: 1.2320838749408722

Best epoch:
epoch         10.000000
train_loss     1.115997
train_acc      0.502000
val_loss       1.232084
val_acc        0.536000
Name: 9, dtype: float64

Saved:
/kaggle/working/results/fs_gsc_smoke/best_model_state.pt
/kaggle/working/results/fs_gsc_smoke/training_history.csv


In [67]:
# ============================================================
# CELL 21 — OFFICIAL FS-GSC BASELINE TRAINING
# TCResNet8 + Prototypical Network
# 5-way 1-shot 5-query
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import random
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm

from protonets.data.FewShotSpeechData import FewShotSpeechDataset
from protonets.models.encoder.TCResNet import TCResNet8
from protonets.models.few_shot import Protonet


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. TORCHAUDIO COMPATIBILITY PATCH
# ============================================================

def load_audio_fixed(self, key, out_field, d):

    sound, _ = torchaudio.load(
        d[key],
        num_frames=self.desired_samples
    )

    d[out_field] = sound

    return d


FewShotSpeechDataset.load_audio = load_audio_fixed


# ============================================================
# 3. CONFIG
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

DATA_ROOT = Path(
    "/kaggle/working/Few-Shot-KWS/speech_commands/core"
)

TRAIN_FILE = DATA_ROOT / "train.txt"
VAL_FILE   = DATA_ROOT / "val.txt"

# Few-shot configuration
N_WAY   = 5
N_SHOT  = 1
N_QUERY = 5

# Official baseline training
MAX_EPOCHS = 30

TRAIN_EPISODES = 50
VAL_EPISODES   = 25

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0

# Stop if validation does not improve
PATIENCE = 8


# ============================================================
# 4. SPEECH CONFIG
# ============================================================

speech_args = {

    "sample_rate": 16000,

    "clip_duration": 1000,

    "window_size": 40,

    "window_stride": 20,

    "num_features": 40,

    "foreground_volume": 1.0,

    # same augmentation used in previous smoke test
    "time_shift": 100,

    # controlled baseline
    "include_background": False,

    "bg_volume": 0.1,

    "bg_frequency": 1.0,

    "include_silence": False,

    "num_silence": 1000,

    "include_unknown": False,
}


# ============================================================
# 5. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/results/fs_gsc_baseline"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL = (
    OUTPUT_DIR /
    "best_model_state.pt"
)

LAST_MODEL = (
    OUTPUT_DIR /
    "last_checkpoint.pt"
)

HISTORY_FILE = (
    OUTPUT_DIR /
    "training_history.csv"
)

CONFIG_FILE = (
    OUTPUT_DIR /
    "config.json"
)


# ============================================================
# 6. SAVE CONFIG
# ============================================================

config = {

    "seed": SEED,

    "device": str(DEVICE),

    "architecture":
        "TCResNet8 + Prototypical Network",

    "embedding_dim": 48,

    "input_shape":
        [1, 51, 40],

    "way":
        N_WAY,

    "shot":
        N_SHOT,

    "query":
        N_QUERY,

    "max_epochs":
        MAX_EPOCHS,

    "train_episodes_per_epoch":
        TRAIN_EPISODES,

    "val_episodes_per_epoch":
        VAL_EPISODES,

    "optimizer":
        "Adam",

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "patience":
        PATIENCE,

    "mfcc": {

        "sample_rate": 16000,

        "window_ms": 40,

        "stride_ms": 20,

        "n_mfcc": 40,

        "n_fft": 640,

        "hop_length": 320,
    }
}


with open(
    CONFIG_FILE,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ============================================================
# 7. LOAD DATASETS
# ============================================================

train_dataset = FewShotSpeechDataset(

    data_dir=str(DATA_ROOT),

    class_file=str(TRAIN_FILE),

    n_support=N_SHOT,

    n_query=N_QUERY,

    cuda=False,

    args=speech_args
)


val_dataset = FewShotSpeechDataset(

    data_dir=str(DATA_ROOT),

    class_file=str(VAL_FILE),

    n_support=N_SHOT,

    n_query=N_QUERY,

    cuda=False,

    args=speech_args
)


print("=" * 80)
print("OFFICIAL FS-GSC BASELINE")
print("=" * 80)

print("Device:", DEVICE)

print(
    "Train classes:",
    len(train_dataset.class_names)
)

print(
    "Validation classes:",
    len(val_dataset.class_names)
)

print()

print(
    f"{N_WAY}-way "
    f"{N_SHOT}-shot "
    f"{N_QUERY}-query"
)

print(
    "Train episodes/epoch:",
    TRAIN_EPISODES
)

print(
    "Validation episodes/epoch:",
    VAL_EPISODES
)

print(
    "Maximum epochs:",
    MAX_EPOCHS
)


# ============================================================
# 8. EPISODE SAMPLER
# ============================================================

def sample_episode(
    dataset,
    n_way
):

    selected = random.sample(
        range(
            len(dataset.class_names)
        ),
        n_way
    )

    items = [
        dataset[i]
        for i in selected
    ]

    xs = torch.stack(
        [
            item["xs"]
            for item in items
        ],
        dim=0
    )

    xq = torch.stack(
        [
            item["xq"]
            for item in items
        ],
        dim=0
    )

    return {

        "xs":
            xs.to(
                DEVICE,
                non_blocking=True
            ),

        "xq":
            xq.to(
                DEVICE,
                non_blocking=True
            )
    }


# ============================================================
# 9. MODEL
# ============================================================

encoder = TCResNet8(

    in_c=1,

    in_h=51,

    in_w=40
)


model = Protonet(

    encoder=encoder,

    encoding="TCResNet8"

).to(DEVICE)


optimizer = torch.optim.Adam(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


parameter_count = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    "\nTrainable parameters:",
    f"{parameter_count:,}"
)


# ============================================================
# 10. RUN EPISODES
# ============================================================

def run_episodes(
    dataset,
    num_episodes,
    training
):

    if training:

        model.train()

    else:

        model.eval()


    losses = []

    accuracies = []


    progress = tqdm(

        range(num_episodes),

        leave=False,

        desc=(
            "train"
            if training
            else "valid"
        )
    )


    for _ in progress:

        sample = sample_episode(

            dataset,

            N_WAY
        )


        if training:

            optimizer.zero_grad()


            loss, output = model.loss(

                sample
            )


            loss.backward()


            optimizer.step()


        else:

            with torch.no_grad():

                loss, output = model.loss(

                    sample
                )


        losses.append(

            float(
                output["loss"]
            )
        )


        accuracies.append(

            float(
                output["acc"]
            )
        )


        progress.set_postfix(

            loss=
                f"{np.mean(losses):.4f}",

            acc=
                f"{np.mean(accuracies):.3f}"
        )


    return (

        float(
            np.mean(losses)
        ),

        float(
            np.mean(accuracies)
        )
    )


# ============================================================
# 11. TRAINING LOOP
# ============================================================

history = []

best_val_loss = float("inf")

best_val_acc = 0.0

best_epoch = 0

epochs_without_improvement = 0


training_start = time.time()


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()


    # ----------------------
    # TRAIN
    # ----------------------

    train_loss, train_acc = (

        run_episodes(

            train_dataset,

            TRAIN_EPISODES,

            training=True
        )
    )


    # ----------------------
    # VALIDATE
    # ----------------------

    val_loss, val_acc = (

        run_episodes(

            val_dataset,

            VAL_EPISODES,

            training=False
        )
    )


    epoch_time = (

        time.time()
        -
        epoch_start
    )


    row = {

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_acc":
            train_acc,

        "val_loss":
            val_loss,

        "val_acc":
            val_acc,

        "epoch_seconds":
            epoch_time
    }


    history.append(row)


    # Save history every epoch
    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_FILE,
        index=False
    )


    print(
        f"Epoch {epoch:02d} | "
        f"train loss={train_loss:.4f} "
        f"acc={train_acc:.3f} | "
        f"val loss={val_loss:.4f} "
        f"acc={val_acc:.3f} | "
        f"{epoch_time:.1f}s"
    )


    # ========================================================
    # SAVE LAST CHECKPOINT EVERY EPOCH
    # ========================================================

    torch.save(

        {

            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "train_loss":
                train_loss,

            "train_acc":
                train_acc,

            "val_loss":
                val_loss,

            "val_acc":
                val_acc,

            "config":
                config
        },

        LAST_MODEL
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_val_acc = val_acc

        best_epoch = epoch

        epochs_without_improvement = 0


        torch.save(

            {

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "train_loss":
                    train_loss,

                "train_acc":
                    train_acc,

                "val_loss":
                    val_loss,

                "val_acc":
                    val_acc,

                "config":
                    config
            },

            BEST_MODEL
        )


        print(
            "   ✅ NEW BEST MODEL"
        )


    else:

        epochs_without_improvement += 1


        print(
            "   No improvement:",
            f"{epochs_without_improvement}/{PATIENCE}"
        )


    # ========================================================
    # EARLY STOP
    # ========================================================

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            "\n🛑 Early stopping triggered."
        )

        break


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

total_time = (

    time.time()
    -
    training_start
)


history_df = pd.DataFrame(
    history
)


print("\n" + "=" * 80)

print(
    "OFFICIAL BASELINE TRAINING COMPLETE"
)

print("=" * 80)


print(
    "Completed epochs:",
    len(history_df)
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    round(best_val_loss, 6)
)

print(
    "Validation accuracy at best loss:",
    round(best_val_acc, 4)
)


best_acc_row = (

    history_df.loc[
        history_df[
            "val_acc"
        ].idxmax()
    ]
)


print("\nHighest validation accuracy:")

print(
    f"{best_acc_row['val_acc']:.4f}"
)

print(
    "at epoch:",
    int(
        best_acc_row["epoch"]
    )
)


print(
    "\nTotal training time:",
    round(
        total_time / 60,
        2
    ),
    "minutes"
)


print("\nFiles saved:")

print(
    "-",
    BEST_MODEL
)

print(
    "-",
    LAST_MODEL
)

print(
    "-",
    HISTORY_FILE
)

print(
    "-",
    CONFIG_FILE
)

/kaggle/working/Few-Shot-KWS
OFFICIAL FS-GSC BASELINE
Device: cuda
Train classes: 20
Validation classes: 5

5-way 1-shot 5-query
Train episodes/epoch: 50
Validation episodes/epoch: 25
Maximum epochs: 30

Trainable parameters: 64,560


train:   0%|          | 0/50 [00:00<?, ?it/s]

valid:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train loss=1.5675 acc=0.283 | val loss=1.5654 acc=0.259 | 114.2s
   ✅ NEW BEST MODEL


train:   0%|          | 0/50 [00:00<?, ?it/s]

...

valid:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 22 | train loss=0.6320 acc=0.756 | val loss=1.3905 acc=0.435 | 0.4s
   No improvement: 7/8


train:   0%|          | 0/50 [00:00<?, ?it/s]

valid:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 23 | train loss=0.5997 acc=0.758 | val loss=1.4364 acc=0.437 | 0.5s
   No improvement: 8/8

🛑 Early stopping triggered.

OFFICIAL BASELINE TRAINING COMPLETE
Completed epochs: 23
Best epoch: 15
Best validation loss: 1.149634
Validation accuracy at best loss: 0.5648

Highest validation accuracy:
0.5648
at epoch: 15

Total training time: 2.08 minutes

Files saved:
- /kaggle/working/results/fs_gsc_baseline/best_model_state.pt
- /kaggle/working/results/fs_gsc_baseline/last_checkpoint.pt
- /kaggle/working/results/fs_gsc_baseline/training_history.csv
- /kaggle/working/results/fs_gsc_baseline/config.json


In [68]:
# ============================================================
# CELL 22 — FINAL FS-GSC UNSEEN-CLASS TEST
# Best TCResNet8 + ProtoNet checkpoint
# 5-way 1-shot 5-query
# ============================================================

%cd /kaggle/working/Few-Shot-KWS

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm

from protonets.data.FewShotSpeechData import FewShotSpeechDataset
from protonets.models.encoder.TCResNet import TCResNet8
from protonets.models.few_shot import Protonet


# ============================================================
# 1. CONFIG
# ============================================================

SEED = 5678

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

DATA_ROOT = Path(
    "/kaggle/working/Few-Shot-KWS/speech_commands/core"
)

TEST_FILE = DATA_ROOT / "test.txt"

CHECKPOINT_PATH = Path(
    "/kaggle/working/results/"
    "fs_gsc_baseline/best_model_state.pt"
)

OUTPUT_DIR = Path(
    "/kaggle/working/results/fs_gsc_baseline"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. TEST CONFIG
# ============================================================

N_WAY = 5
N_SHOT = 1
N_QUERY = 5

# More episodes => more stable estimate
TEST_EPISODES = 500


# ============================================================
# 3. TORCHAUDIO PATCH
# ============================================================

def load_audio_fixed(self, key, out_field, d):

    sound, _ = torchaudio.load(
        d[key],
        num_frames=self.desired_samples
    )

    d[out_field] = sound

    return d


FewShotSpeechDataset.load_audio = load_audio_fixed


# ============================================================
# 4. SPEECH CONFIG — SAME AS TRAIN
# ============================================================

speech_args = {

    "sample_rate": 16000,

    "clip_duration": 1000,

    "window_size": 40,

    "window_stride": 20,

    "num_features": 40,

    "foreground_volume": 1.0,

    "time_shift": 100,

    "include_background": False,

    "bg_volume": 0.1,

    "bg_frequency": 1.0,

    "include_silence": False,

    "num_silence": 1000,

    "include_unknown": False,
}


# ============================================================
# 5. LOAD TEST DATASET
# ============================================================

test_dataset = FewShotSpeechDataset(

    data_dir=str(DATA_ROOT),

    class_file=str(TEST_FILE),

    n_support=N_SHOT,

    n_query=N_QUERY,

    cuda=False,

    args=speech_args
)


print("=" * 80)
print("FS-GSC FINAL TEST")
print("=" * 80)

print("Device:", DEVICE)

print(
    "Test classes:",
    len(test_dataset.class_names)
)

print(
    "Classes:",
    test_dataset.class_names
)

print(
    f"\nEpisode: "
    f"{N_WAY}-way "
    f"{N_SHOT}-shot "
    f"{N_QUERY}-query"
)

print(
    "Test episodes:",
    TEST_EPISODES
)


# ============================================================
# 6. BUILD MODEL
# ============================================================

encoder = TCResNet8(
    in_c=1,
    in_h=51,
    in_w=40
)

model = Protonet(
    encoder=encoder,
    encoding="TCResNet8"
).to(DEVICE)


# ============================================================
# 7. LOAD BEST CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()


print("\nLoaded checkpoint:")
print(CHECKPOINT_PATH)

print(
    "Best training epoch:",
    checkpoint["epoch"]
)

print(
    "Checkpoint val loss:",
    checkpoint["val_loss"]
)

print(
    "Checkpoint val acc:",
    checkpoint["val_acc"]
)


# ============================================================
# 8. SAMPLE TEST EPISODE
# ============================================================

def sample_episode(dataset):

    # There are exactly 5 test classes.
    # Still shuffle order every episode.
    selected = random.sample(
        range(len(dataset.class_names)),
        N_WAY
    )

    items = [
        dataset[i]
        for i in selected
    ]

    xs = torch.stack(
        [
            item["xs"]
            for item in items
        ],
        dim=0
    )

    xq = torch.stack(
        [
            item["xq"]
            for item in items
        ],
        dim=0
    )

    return {

        "xs":
            xs.to(
                DEVICE,
                non_blocking=True
            ),

        "xq":
            xq.to(
                DEVICE,
                non_blocking=True
            )
    }


# ============================================================
# 9. FINAL TEST
# ============================================================

losses = []
accuracies = []

progress = tqdm(
    range(TEST_EPISODES),
    desc="Final test"
)

with torch.no_grad():

    for _ in progress:

        sample = sample_episode(
            test_dataset
        )

        loss, output = model.loss(
            sample
        )

        losses.append(
            float(output["loss"])
        )

        accuracies.append(
            float(output["acc"])
        )

        progress.set_postfix(
            loss=f"{np.mean(losses):.4f}",
            acc=f"{np.mean(accuracies):.4f}"
        )


# ============================================================
# 10. STATISTICS
# ============================================================

losses = np.array(losses)

accuracies = np.array(
    accuracies
)

mean_loss = float(
    losses.mean()
)

mean_acc = float(
    accuracies.mean()
)

std_acc = float(
    accuracies.std(ddof=1)
)

standard_error = (
    std_acc
    /
    np.sqrt(TEST_EPISODES)
)

ci95 = (
    1.96
    *
    standard_error
)


# ============================================================
# 11. RESULTS
# ============================================================

print("\n" + "=" * 80)
print("FINAL FS-GSC TEST RESULT")
print("=" * 80)

print(
    "Mean test loss:",
    round(mean_loss, 6)
)

print(
    "Mean test accuracy:",
    f"{mean_acc * 100:.2f}%"
)

print(
    "Episode accuracy std:",
    f"{std_acc * 100:.2f}%"
)

print(
    "95% confidence interval:",
    f"± {ci95 * 100:.2f}%"
)

print(
    "\nRandom 5-way baseline:",
    "20.00%"
)


# ============================================================
# 12. SAVE RESULTS
# ============================================================

episode_df = pd.DataFrame({

    "episode":
        np.arange(
            1,
            TEST_EPISODES + 1
        ),

    "loss":
        losses,

    "accuracy":
        accuracies
})

episode_df.to_csv(
    OUTPUT_DIR /
    "final_test_episodes.csv",
    index=False
)


summary = pd.DataFrame([
    {
        "way":
            N_WAY,

        "shot":
            N_SHOT,

        "query":
            N_QUERY,

        "episodes":
            TEST_EPISODES,

        "mean_loss":
            mean_loss,

        "mean_accuracy":
            mean_acc,

        "accuracy_std":
            std_acc,

        "ci95":
            ci95,

        "checkpoint_epoch":
            checkpoint["epoch"]
    }
])


summary.to_csv(
    OUTPUT_DIR /
    "final_test_summary.csv",
    index=False
)


print("\nSaved:")

print(
    OUTPUT_DIR /
    "final_test_episodes.csv"
)

print(
    OUTPUT_DIR /
    "final_test_summary.csv"
)

/kaggle/working/Few-Shot-KWS
FS-GSC FINAL TEST
Device: cuda
Test classes: 5
Classes: ['one', 'marvin', 'cat', 'two', 'no']

Episode: 5-way 1-shot 5-query
Test episodes: 500

Loaded checkpoint:
/kaggle/working/results/fs_gsc_baseline/best_model_state.pt
Best training epoch: 15
Checkpoint val loss: 1.1496336245536805
Checkpoint val acc: 0.5647999858856201


Final test:   0%|          | 0/500 [00:00<?, ?it/s]


FINAL FS-GSC TEST RESULT
Mean test loss: 1.228017
Mean test accuracy: 50.50%
Episode accuracy std: 12.06%
95% confidence interval: ± 1.06%

Random 5-way baseline: 20.00%

Saved:
/kaggle/working/results/fs_gsc_baseline/final_test_episodes.csv
/kaggle/working/results/fs_gsc_baseline/final_test_summary.csv


In [69]:
# ============================================================
# CELL 23 — AUDIT ORIGINAL PAPER EXPERIMENT SCRIPTS
# ============================================================

from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

files = [
    REPO / "scripts/train/few_shot/fewshotspeech/train.sh",
    REPO / "scripts/train/few_shot/fewshotspeech/train_fewshotspeech.sh",
    REPO / "scripts/train/few_shot/fewshotspeech/qsub_train.sh",
    REPO / "scripts/train/few_shot/fewshotspeech/loop_qsub.sh",
]

for path in files:

    print("\n" + "=" * 100)
    print(path.relative_to(REPO))
    print("=" * 100)

    if not path.exists():
        print("NOT FOUND")
        continue

    print(path.read_text())


scripts/train/few_shot/fewshotspeech/train.sh
way=$1
shot=$2
i=$3
id=$4

args="../run_train.py \
		--data.dataset=googlespeech --data.way=$way --data.shot=$shot \
		--data.query=5 --data.test_way=$way --data.test_shot=$shot --data.test_query=15 \
		--data.train_episodes=200 --data.test_episodes=100 \
		--model.model_name=protonet_conv --model.x_dim=1,51,40 --model.hid_dim=64 --model.z_dim=64 \
		--train.epochs=200 --train.optim_method=Adam --train.learning_rate=0.001 --train.decay_every=20 \
		--train.weight_decay=0.0 --train.patience=200  \
		--log.exp_dir=./$id \
		--data.cuda \
        --model.encoding=TCResNet8Dilated"

if [ "$i" -eq "1" ]; then
    args+=" --speech.include_background"

elif [ "$i" -eq "2" ]; then
    args+=" --speech.include_silence"

elif [ "$i" -eq "3" ]; then
    args+=" --speech.include_unknown"

elif [ "$i" -eq "4" ]; then
    args+=" --speech.include_background"
    args+=" --speech.include_silence"

elif [ "$i" -eq "5" ]; then
    args+=" --speech.include_

In [11]:
# ============================================================
# CELL 24 — BUILD FULL PAPER EXPERIMENT MANIFEST
# ============================================================

from pathlib import Path
import pandas as pd

WAYS = [2, 3, 4, 5]

SHOTS = [
    1,
    5,
    10,
    15,
    20
]

conditions = {

    0: {
        "background": False,
        "silence": False,
        "unknown": False
    },

    1: {
        "background": True,
        "silence": False,
        "unknown": False
    },

    2: {
        "background": False,
        "silence": True,
        "unknown": False
    },

    3: {
        "background": False,
        "silence": False,
        "unknown": True
    },

    4: {
        "background": True,
        "silence": True,
        "unknown": False
    },

    5: {
        "background": True,
        "silence": False,
        "unknown": True
    },

    6: {
        "background": False,
        "silence": True,
        "unknown": True
    },

    7: {
        "background": True,
        "silence": True,
        "unknown": True
    }
}


rows = []

exp_id = 1

for way in WAYS:

    for shot in SHOTS:

        for flag in range(8):

            c = conditions[flag]

            rows.append({

                "experiment_id":
                    exp_id,

                "way":
                    way,

                "shot":
                    shot,

                "flag":
                    flag,

                "background":
                    c["background"],

                "silence":
                    c["silence"],

                "unknown":
                    c["unknown"],

                "train_query":
                    5,

                "test_query":
                    15,

                "train_episodes":
                    200,

                "test_episodes":
                    100,

                "epochs":
                    200,

                "encoder":
                    "TCResNet8Dilated",

                "optimizer":
                    "Adam",

                "learning_rate":
                    0.001,

                "decay_every":
                    20,

                "weight_decay":
                    0.0,

                "patience":
                    200,

                "status":
                    "pending"
            })

            exp_id += 1


manifest = pd.DataFrame(rows)


OUT_DIR = Path(
    "/kaggle/working/results/"
    "paper_full_reproduction"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST = (
    OUT_DIR /
    "experiment_manifest.csv"
)


manifest.to_csv(
    MANIFEST,
    index=False
)


print("=" * 80)

print(
    "FULL PAPER EXPERIMENT MANIFEST"
)

print("=" * 80)

print(
    "Total experiments:",
    len(manifest)
)

print(
    "\nWays:",
    sorted(
        manifest["way"].unique()
    )
)

print(
    "Shots:",
    sorted(
        manifest["shot"].unique()
    )
)

print(
    "Conditions:",
    manifest["flag"].nunique()
)


print("\nExperiments per way:")

print(
    manifest.groupby(
        "way"
    ).size()
)


print(
    "\nExperiments per shot:"
)

print(
    manifest.groupby(
        "shot"
    ).size()
)


print(
    "\nFirst 20 experiments:"
)

print(
    manifest.head(20).to_string(
        index=False
    )
)


print("\nSaved:")
print(MANIFEST)

FULL PAPER EXPERIMENT MANIFEST
Total experiments: 160

Ways: [np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Shots: [np.int64(1), np.int64(5), np.int64(10), np.int64(15), np.int64(20)]
Conditions: 8

Experiments per way:
way
2    40
3    40
4    40
5    40
dtype: int64

Experiments per shot:
shot
1     32
5     32
10    32
15    32
20    32
dtype: int64

First 20 experiments:
 experiment_id  way  shot  flag  background  silence  unknown  train_query  test_query  train_episodes  test_episodes  epochs          encoder optimizer  learning_rate  decay_every  weight_decay  patience  status
             1    2     1     0       False    False    False            5          15             200            100     200 TCResNet8Dilated      Adam          0.001           20           0.0       200 pending
             2    2     1     1        True    False    False            5          15             200            100     200 TCResNet8Dilated      Adam          0.001           20          

In [8]:
# ============================================================
# CELL 25A — COMPATIBILITY PATCH FOR ORIGINAL PAPER PIPELINE
# ============================================================

from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

# ------------------------------------------------------------
# 1. Patch torchaudio.load old API
# ------------------------------------------------------------

data_file = REPO / "protonets/data/FewShotSpeechData.py"

text = data_file.read_text()

old = """sound, _ = torchaudio.load(filepath=d[key], normalization=True,
                                         num_frames=self.desired_samples)"""

new = """sound, _ = torchaudio.load(
            d[key],
            num_frames=self.desired_samples
        )"""

if old in text:
    text = text.replace(old, new)
    data_file.write_text(text)
    print("✅ Patched FewShotSpeechData torchaudio.load")
else:
    print("ℹ️ FewShotSpeechData patch already applied / source differs")


# ------------------------------------------------------------
# 2. Patch evaluation torch.load for PyTorch >= 2.6
# ------------------------------------------------------------

eval_file = REPO / "scripts/predict/few_shot/eval.py"

text = eval_file.read_text()

old = "model = torch.load(opt['model.model_path'])"
new = "model = torch.load(opt['model.model_path'], weights_only=False)"

if old in text:
    text = text.replace(old, new)

# Python 3 compatibility
old2 = """model_opt['model.x_dim'] = map(int, model_opt['model.x_dim'].split(','))"""
new2 = """model_opt['model.x_dim'] = list(map(int, model_opt['model.x_dim'].split(',')))"""

if old2 in text:
    text = text.replace(old2, new2)

eval_file.write_text(text)

print("✅ Patched eval.py compatibility")


# ------------------------------------------------------------
# 3. Verify important files
# ------------------------------------------------------------

required = [
    REPO / "scripts/train/few_shot/run_train.py",
    REPO / "scripts/train/few_shot/train.py",
    REPO / "scripts/predict/few_shot/run_eval.py",
    REPO / "speech_commands/core/train.txt",
    REPO / "speech_commands/core/val.txt",
    REPO / "speech_commands/core/test.txt",
]

print("\nVERIFY:")

for p in required:
    print(
        "✅" if p.exists() else "❌",
        p.relative_to(REPO)
    )

ℹ️ FewShotSpeechData patch already applied / source differs
✅ Patched eval.py compatibility

VERIFY:
✅ scripts/train/few_shot/run_train.py
✅ scripts/train/few_shot/train.py
✅ scripts/predict/few_shot/run_eval.py
✅ speech_commands/core/train.txt
✅ speech_commands/core/val.txt
✅ speech_commands/core/test.txt


In [75]:
# ============================================================
# CELL 25B — FULL PAPER EXPERIMENT RUNNER
#
# Uses ORIGINAL:
# scripts/train/few_shot/run_train.py
#
# Resume via experiment_manifest.csv
# ============================================================

import os
import sys
import json
import time
import subprocess
from pathlib import Path

import pandas as pd


# ============================================================
# PATHS
# ============================================================

REPO = Path(
    "/kaggle/working/Few-Shot-KWS"
)

MASTER_DIR = Path(
    "/kaggle/working/results/paper_full_reproduction"
)

MASTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_PATH = (
    MASTER_DIR /
    "experiment_manifest.csv"
)


if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest missing: {MANIFEST_PATH}"
    )


# ============================================================
# IMPORTANT
#
# FIRST RUN:
#   RUN_LIMIT = 1
#
# After exp 1 works:
#   RUN_LIMIT = 5
# or 10
#
# None = run every remaining experiment
# ============================================================

RUN_LIMIT = 1


# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH
)

# Ensure extra bookkeeping columns exist
extra_columns = {

    "train_status": "pending",

    "eval_status": "pending",

    "result_dir": "",

    "return_code": "",

    "elapsed_minutes": "",

    "error": "",
}


for column, default in extra_columns.items():

    if column not in manifest.columns:

        manifest[column] = default


# ============================================================
# CONDITION FLAGS
# ============================================================

def condition_args(row):

    args = []

    if bool(row["background"]):
        args.append(
            "--speech.include_background"
        )

    if bool(row["silence"]):
        args.append(
            "--speech.include_silence"
        )

    if bool(row["unknown"]):
        args.append(
            "--speech.include_unknown"
        )

    return args


# ============================================================
# FIND RESULT DIRECTORY CREATED BY TRAIN.PY
# ============================================================

def newest_timestamp_dir(parent):

    if not parent.exists():
        return None

    dirs = [
        p for p in parent.iterdir()
        if p.is_dir()
    ]

    if not dirs:
        return None

    return max(
        dirs,
        key=lambda p: p.stat().st_mtime
    )


# ============================================================
# PENDING EXPERIMENTS
# ============================================================

pending_indices = [

    idx

    for idx, row
    in manifest.iterrows()

    if row["train_status"] != "done"
]


if RUN_LIMIT is not None:
    pending_indices = (
        pending_indices[:RUN_LIMIT]
    )


print("=" * 90)
print("PAPER FULL REPRODUCTION RUNNER")
print("=" * 90)

print(
    "Total experiments:",
    len(manifest)
)

print(
    "Already trained:",
    int(
        (
            manifest["train_status"]
            ==
            "done"
        ).sum()
    )
)

print(
    "Experiments this run:",
    len(pending_indices)
)


# ============================================================
# RUN
# ============================================================

for manifest_idx in pending_indices:

    row = manifest.loc[
        manifest_idx
    ]

    exp_id = int(
        row["experiment_id"]
    )

    way = int(
        row["way"]
    )

    shot = int(
        row["shot"]
    )

    flag = int(
        row["flag"]
    )


    print("\n" + "=" * 90)

    print(
        f"EXPERIMENT {exp_id:03d}/160"
    )

    print("=" * 90)

    print(
        f"way={way}, "
        f"shot={shot}, "
        f"flag={flag}"
    )

    print(
        "background=",
        bool(row["background"])
    )

    print(
        "silence=",
        bool(row["silence"])
    )

    print(
        "unknown=",
        bool(row["unknown"])
    )


    # --------------------------------------------------------
    # Separate directory per experiment
    #
    # train.py automatically creates:
    #
    # REPO/results/<log.exp_dir>/<timestamp>
    # --------------------------------------------------------

    exp_name = (
        f"paper_full_reproduction/"
        f"exp_{exp_id:03d}"
    )

    paper_result_parent = (
        REPO /
        "results" /
        "paper_full_reproduction" /
        f"exp_{exp_id:03d}"
    )


    # --------------------------------------------------------
    # EXACT PAPER COMMAND
    # --------------------------------------------------------

    cmd = [

        sys.executable,

        "scripts/train/few_shot/run_train.py",

        "--data.dataset=googlespeech",

        f"--data.way={way}",

        f"--data.shot={shot}",

        "--data.query=5",

        f"--data.test_way={way}",

        f"--data.test_shot={shot}",

        "--data.test_query=15",

        "--data.train_episodes=200",

        "--data.test_episodes=100",

        "--model.model_name=protonet_conv",

        "--model.x_dim=1,51,40",

        "--model.hid_dim=64",

        "--model.z_dim=64",

        "--model.encoding=TCResNet8Dilated",

        "--train.epochs=200",

        "--train.optim_method=Adam",

        "--train.learning_rate=0.001",

        "--train.decay_every=20",

        "--train.weight_decay=0.0",

        "--train.patience=200",

        f"--log.exp_dir={exp_name}",

        "--data.cuda",
    ]


    cmd.extend(
        condition_args(row)
    )


    print("\nCOMMAND:")

    print(
        " ".join(cmd)
    )


    # --------------------------------------------------------
    # STATUS = RUNNING
    # --------------------------------------------------------

    manifest.loc[
        manifest_idx,
        "train_status"
    ] = "running"

    manifest.loc[
        manifest_idx,
        "error"
    ] = ""

    manifest.to_csv(
        MANIFEST_PATH,
        index=False
    )


    start = time.time()


    # --------------------------------------------------------
    # TRAIN USING ORIGINAL PAPER CODE
    # --------------------------------------------------------

    try:

        process = subprocess.run(

            cmd,

            cwd=str(REPO),

            check=False
        )


        elapsed = (
            time.time()
            -
            start
        ) / 60


        manifest.loc[
            manifest_idx,
            "return_code"
        ] = process.returncode

        manifest.loc[
            manifest_idx,
            "elapsed_minutes"
        ] = elapsed


        # ----------------------------------------------------
        # Locate generated timestamp directory
        # ----------------------------------------------------

        result_dir = newest_timestamp_dir(
            paper_result_parent
        )


        if result_dir is not None:

            manifest.loc[
                manifest_idx,
                "result_dir"
            ] = str(
                result_dir
            )


        # ----------------------------------------------------
        # SUCCESS CHECK
        # ----------------------------------------------------

        best_model = (

            result_dir /
            "best_model.pt"

            if result_dir
            else None
        )


        trace_file = (

            result_dir /
            "trace.txt"

            if result_dir
            else None
        )


        opt_file = (

            result_dir /
            "opt.json"

            if result_dir
            else None
        )


        success = (

            process.returncode == 0

            and best_model is not None

            and best_model.exists()

            and trace_file.exists()

            and opt_file.exists()
        )


        if success:

            manifest.loc[
                manifest_idx,
                "train_status"
            ] = "done"

            manifest.loc[
                manifest_idx,
                "status"
            ] = "trained"

            print(
                "\n✅ TRAINING COMPLETE"
            )

            print(
                "Result:",
                result_dir
            )

            print(
                "Elapsed:",
                round(elapsed, 2),
                "min"
            )


        else:

            manifest.loc[
                manifest_idx,
                "train_status"
            ] = "failed"

            manifest.loc[
                manifest_idx,
                "status"
            ] = "failed"

            manifest.loc[
                manifest_idx,
                "error"
            ] = (
                "Training process failed "
                "or expected files missing"
            )

            print(
                "\n❌ EXPERIMENT FAILED"
            )


    except Exception as e:

        manifest.loc[
            manifest_idx,
            "train_status"
        ] = "failed"

        manifest.loc[
            manifest_idx,
            "status"
        ] = "failed"

        manifest.loc[
            manifest_idx,
            "error"
        ] = repr(e)

        print(
            "\n❌ EXCEPTION:"
        )

        print(e)


    # --------------------------------------------------------
    # SAVE MANIFEST AFTER EVERY EXPERIMENT
    # --------------------------------------------------------

    manifest.to_csv(
        MANIFEST_PATH,
        index=False
    )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("RUN COMPLETE")
print("=" * 90)

print(
    manifest[
        "train_status"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nManifest:"
)

print(
    MANIFEST_PATH
)

PAPER FULL REPRODUCTION RUNNER
Total experiments: 160
Already trained: 0
Experiments this run: 1

EXPERIMENT 001/160
way=2, shot=1, flag=0
background= False
silence= False
unknown= False

COMMAND:
/usr/bin/python3 scripts/train/few_shot/run_train.py --data.dataset=googlespeech --data.way=2 --data.shot=1 --data.query=5 --data.test_way=2 --data.test_shot=1 --data.test_query=15 --data.train_episodes=200 --data.test_episodes=100 --model.model_name=protonet_conv --model.x_dim=1,51,40 --model.hid_dim=64 --model.z_dim=64 --model.encoding=TCResNet8Dilated --train.epochs=200 --train.optim_method=Adam --train.learning_rate=0.001 --train.decay_every=20 --train.weight_decay=0.0 --train.patience=200 --log.exp_dir=paper_full_reproduction/exp_001 --data.cuda


/kaggle/working/Few-Shot-KWS/scripts/train/few_shot/train.py:73: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  state['scheduler'].step()
Epoch 2 train:   9%|▉         | 18/200 [00:00<00:01, 172.13it/s]

Epoch 01: train loss = 0.958869, train acc = 0.605500, val loss = 0.659432, val acc = 0.655000
==> best model (loss = 0.659432), saving model...


Epoch 3 train:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 02: train loss = 0.540972, train acc = 0.743000, val loss = 0.611939, val acc = 0.669000
==> best model (loss = 0.611939), saving model...


...

Epoch 198: train loss = 0.089849, train acc = 0.966000, val loss = 0.347896, val acc = 0.854000


Epoch 200 train:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 199: train loss = 0.078846, train acc = 0.969000, val loss = 0.348606, val acc = 0.846000


Epoch 200 valid: 100%|██████████| 100/100 [00:00<00:00, 420.38it/s]


Epoch 200: train loss = 0.084285, train acc = 0.965500, val loss = 0.290621, val acc = 0.881000
Total Time: 0:06:22.900150

✅ TRAINING COMPLETE
Result: /kaggle/working/Few-Shot-KWS/results/paper_full_reproduction/exp_001/2026-10-01 15:36:41
Elapsed: 6.5 min

RUN COMPLETE
train_status
pending    159
done         1
Name: count, dtype: int64

Manifest:
/kaggle/working/results/paper_full_reproduction/experiment_manifest.csv


In [9]:
from pathlib import Path

REPO = Path("/kaggle/working/Few-Shot-KWS")

file = REPO / "protonets/utils/data.py"

text = file.read_text()

text = text.replace(
    "../../data/speech_commands",
    "../../speech_commands"
)

file.write_text(text)

print("✅ fixed data path")

✅ fixed data path


In [10]:
from pathlib import Path

root = Path(
    "/kaggle/working/Few-Shot-KWS/speech_commands"
)

for p in [
    root / "core/train.txt",
    root / "core/val.txt",
    root / "core/test.txt",
    root / "_unknown_",
    root / "_background_noise_",
]:
    print("✅" if p.exists() else "❌", p)

✅ /kaggle/working/Few-Shot-KWS/speech_commands/core/train.txt
✅ /kaggle/working/Few-Shot-KWS/speech_commands/core/val.txt
✅ /kaggle/working/Few-Shot-KWS/speech_commands/core/test.txt
✅ /kaggle/working/Few-Shot-KWS/speech_commands/_unknown_
✅ /kaggle/working/Few-Shot-KWS/speech_commands/_background_noise_


In [76]:
# ============================================================
# CELL 26A — LOCATE EXP001 RESULT
# ============================================================

import pandas as pd
from pathlib import Path

manifest_path = Path(
    "/kaggle/working/results/"
    "paper_full_reproduction/"
    "experiment_manifest.csv"
)

df = pd.read_csv(manifest_path)

row = df.loc[
    df["experiment_id"] == 1
].iloc[0]

print("=" * 80)
print("EXP001")
print("=" * 80)

print(row.to_string())

result_dir = Path(row["result_dir"])

print("\nResult dir:")
print(result_dir)

print("\nFiles:")

for p in sorted(result_dir.iterdir()):
    print("-", p.name)

EXP001
experiment_id                                                      1
way                                                                2
shot                                                               1
flag                                                               0
background                                                     False
silence                                                        False
unknown                                                        False
train_query                                                        5
test_query                                                        15
train_episodes                                                   200
test_episodes                                                    100
epochs                                                           200
encoder                                             TCResNet8Dilated
optimizer                                                       Adam
learning_rate              

In [77]:
# ============================================================
# CELL 26B — EVALUATE EXP001 USING ORIGINAL PAPER SCRIPT
# ============================================================

import subprocess
import sys
from pathlib import Path

best_model = result_dir / "best_model.pt"

if not best_model.exists():
    raise FileNotFoundError(best_model)

cmd = [
    sys.executable,
    "scripts/predict/few_shot/run_eval.py",
    f"--model.model_path={best_model}",
]

print("COMMAND:")
print(" ".join(map(str, cmd)))

process = subprocess.run(
    cmd,
    cwd="/kaggle/working/Few-Shot-KWS",
    check=False
)

print("\nReturn code:", process.returncode)

COMMAND:
/usr/bin/python3 scripts/predict/few_shot/run_eval.py --model.model_path=/kaggle/working/Few-Shot-KWS/results/paper_full_reproduction/exp_001/2026-10-01 15:36:41/best_model.pt
Evaluating 2-way, 1-shot with 15 query examples/class over 100 episodes


test: 100%|██████████| 100/100 [00:23<00:00,  4.28it/s]



Return code: 0


In [78]:
# ============================================================
# CELL 26C — READ EXP001 EVAL RESULT
# ============================================================

import json

eval_file = result_dir / "eval.txt"

if not eval_file.exists():
    raise FileNotFoundError(eval_file)

with open(eval_file, "r") as f:
    result = json.load(f)

print("=" * 80)
print("EXP001 FINAL EVALUATION")
print("=" * 80)

print(json.dumps(result, indent=2))

EXP001 FINAL EVALUATION
{
  "test": {
    "loss": {
      "mean": 0.32201096110045907,
      "confidence": 0.04871152040887154
    },
    "acc": {
      "mean": 0.8540000399947165,
      "confidence": 0.027543508388039567
    }
  }
}


In [12]:
# ============================================================
# CELL 26D — PATCH AUDIO CACHE (RAM) FOR FewShotSpeechData
# Chạy SAU Cell 25A, TRƯỚC Cell 27
# ============================================================

from pathlib import Path

f = Path("/kaggle/working/Few-Shot-KWS/protonets/data/FewShotSpeechData.py")
t = f.read_text()

if "_AUDIO_CACHE" in t:
    print("ℹ️ Đã patch cache rồi, bỏ qua.")
else:
    old = """sound, _ = torchaudio.load(
            d[key],
            num_frames=self.desired_samples
        )"""

    new = """path = d[key]
        sound = _AUDIO_CACHE.get(path)
        if sound is None:
            sound, _ = torchaudio.load(path, num_frames=self.desired_samples)
            _AUDIO_CACHE[path] = sound
        sound = sound.clone()"""

    if old not in t:
        raise RuntimeError(
            "Không khớp đoạn load_audio. Hãy chạy Cell 25A trước "
            "rồi in file ra để kiểm tra."
        )

    t = t.replace(old, new)
    t = t.replace(
        "class FewShotSpeechDataset(",
        "_AUDIO_CACHE = {}\n\n\nclass FewShotSpeechDataset(",
        1,
    )
    f.write_text(t)
    print("✅ Đã thêm cache audio vào FewShotSpeechData.py")

# kiểm tra
print("\nXác nhận:")
print("_AUDIO_CACHE có trong file:", "_AUDIO_CACHE" in f.read_text())

✅ Đã thêm cache audio vào FewShotSpeechData.py

Xác nhận:
_AUDIO_CACHE có trong file: True


In [19]:
# ============================================================
# CELL 27 — MULTI-JOB PAPER REPRODUCTION RUNNER (QUEUE)
#
# - Hàng đợi động: GPU nào rảnh thì lấy việc tiếp
# - JOBS_PER_GPU process chạy song song trên mỗi GPU
# - Giới hạn thread CPU mỗi process
# - Train + Eval + Resume + Manifest update (thread-safe)
# ============================================================

import os
import sys
import json
import time
import threading
import subprocess
from pathlib import Path
from queue import Queue, Empty
from concurrent.futures import ThreadPoolExecutor

import pandas as pd


# ============================================================
# PATHS
# ============================================================

REPO = Path("/kaggle/working/Few-Shot-KWS")

MASTER_DIR = Path("/kaggle/working/results/paper_full_reproduction")

MANIFEST_PATH = MASTER_DIR / "experiment_manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(MANIFEST_PATH)


# ============================================================
# CONFIG
# ============================================================

GPU_IDS = [0, 1]

# Số job chạy song song trên MỖI GPU.
# Bắt đầu với 2. Nếu RAM/CPU nghẽn thì hạ về 1; nếu còn dư thì thử 3.
JOBS_PER_GPU = 2

# Resume point: EXP001-EXP051 đã chạy xong và backup ở session trước.
START_EXP_ID = 52

# Số experiment tối đa trong 1 lần chạy notebook (None = chạy hết)
MAX_EXPERIMENTS_THIS_RUN = 24

# Số thread CPU cho mỗi process con
CPU_THREADS_PER_JOB = 1

PYTHON = sys.executable

SLOTS = [g for g in GPU_IDS for _ in range(JOBS_PER_GPU)]


# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(MANIFEST_PATH)

needed_columns = {
    "train_status": "pending",
    "eval_status": "pending",
    "result_dir": "",
    "return_code": "",
    "elapsed_minutes": "",
    "test_loss_mean": "",
    "test_loss_ci95": "",
    "test_acc_mean": "",
    "test_acc_ci95": "",
    "error": "",
}

for col, default in needed_columns.items():
    if col not in manifest.columns:
        manifest[col] = default

# các cột có thể bị pandas đọc thành float -> ép sang object để ghi chuỗi
for col in ["result_dir", "error", "return_code", "train_status", "eval_status"]:
    manifest[col] = manifest[col].astype(object)


# ============================================================
# FIX EXP001 EVAL STATUS IF NEEDED
# ============================================================

exp1_mask = manifest["experiment_id"] == 1

if exp1_mask.any():
    row1 = manifest.loc[exp1_mask].iloc[0]
    result_dir_1 = str(row1.get("result_dir", ""))

    if result_dir_1 and result_dir_1 != "nan":
        eval_file_1 = Path(result_dir_1) / "eval.txt"

        if eval_file_1.exists():
            with open(eval_file_1, "r") as f:
                eval_data = json.load(f)

            manifest.loc[exp1_mask, "eval_status"] = "done"
            manifest.loc[exp1_mask, "test_loss_mean"] = eval_data["test"]["loss"]["mean"]
            manifest.loc[exp1_mask, "test_loss_ci95"] = eval_data["test"]["loss"]["confidence"]
            manifest.loc[exp1_mask, "test_acc_mean"] = eval_data["test"]["acc"]["mean"]
            manifest.loc[exp1_mask, "test_acc_ci95"] = eval_data["test"]["acc"]["confidence"]

manifest.to_csv(MANIFEST_PATH, index=False)


# ============================================================
# HELPERS
# ============================================================

def condition_args(row):
    args = []
    if bool(row["background"]):
        args.append("--speech.include_background")
    if bool(row["silence"]):
        args.append("--speech.include_silence")
    if bool(row["unknown"]):
        args.append("--speech.include_unknown")
    return args


def newest_timestamp_dir(parent):
    if not parent.exists():
        return None

    dirs = [p for p in parent.iterdir() if p.is_dir()]

    if not dirs:
        return None

    return max(dirs, key=lambda p: p.stat().st_mtime)


def build_train_cmd(row):
    exp_id = int(row["experiment_id"])
    way = int(row["way"])
    shot = int(row["shot"])

    exp_name = f"paper_full_reproduction/exp_{exp_id:03d}"

    cmd = [
        PYTHON,
        "scripts/train/few_shot/run_train.py",

        "--data.dataset=googlespeech",
        f"--data.way={way}",
        f"--data.shot={shot}",
        "--data.query=5",
        f"--data.test_way={way}",
        f"--data.test_shot={shot}",
        "--data.test_query=15",
        "--data.train_episodes=200",
        "--data.test_episodes=100",

        "--model.model_name=protonet_conv",
        "--model.x_dim=1,51,40",
        "--model.hid_dim=64",
        "--model.z_dim=64",
        "--model.encoding=TCResNet8Dilated",

        "--train.epochs=200",
        "--train.optim_method=Adam",
        "--train.learning_rate=0.001",
        "--train.decay_every=20",
        "--train.weight_decay=0.0",
        "--train.patience=200",

        f"--log.exp_dir={exp_name}",
        "--data.cuda",
    ]

    cmd.extend(condition_args(row))
    return cmd


# ============================================================
# RUN ONE EXPERIMENT
# ============================================================

def run_experiment(row, gpu_id):
    exp_id = int(row["experiment_id"])
    way = int(row["way"])
    shot = int(row["shot"])
    flag = int(row["flag"])

    print("\n" + "=" * 90)
    print(f"[GPU {gpu_id}] START EXP {exp_id:03d}")
    print(f"[GPU {gpu_id}] {way}-way {shot}-shot flag={flag}")
    print("=" * 90)

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        # để subprocess tìm thấy package protonets ở thư mục gốc repo
    old_pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(REPO) + (os.pathsep + old_pp if old_pp else "")

    n = str(CPU_THREADS_PER_JOB)
    env["OMP_NUM_THREADS"] = n
    env["MKL_NUM_THREADS"] = n
    env["OPENBLAS_NUM_THREADS"] = n
    env["NUMEXPR_NUM_THREADS"] = n

    cmd = build_train_cmd(row)

    start = time.time()

    result = {
        "experiment_id": exp_id,
        "gpu_id": gpu_id,
        "success": False,
        "train_return_code": None,
        "eval_return_code": None,
        "elapsed_minutes": None,
        "result_dir": "",
        "test_loss_mean": None,
        "test_loss_ci95": None,
        "test_acc_mean": None,
        "test_acc_ci95": None,
        "error": "",
    }

    try:
        # ---------------- TRAIN ----------------
        process = subprocess.run(
            cmd, cwd=str(REPO), env=env, check=False
        )

        result["train_return_code"] = process.returncode

        if process.returncode != 0:
            result["error"] = f"train failed return_code={process.returncode}"
            return result

        # ---------------- FIND RESULT DIR ----------------
        parent = (
            REPO / "results" / "paper_full_reproduction" / f"exp_{exp_id:03d}"
        )

        result_dir = newest_timestamp_dir(parent)

        if result_dir is None:
            result["error"] = "result directory not found"
            return result

        result["result_dir"] = str(result_dir)

        best_model = result_dir / "best_model.pt"

        if not best_model.exists():
            result["error"] = "best_model.pt not found"
            return result

        # ---------------- EVAL ----------------
        eval_cmd = [
            PYTHON,
            "scripts/predict/few_shot/run_eval.py",
            f"--model.model_path={best_model}",
        ]

        eval_process = subprocess.run(
            eval_cmd, cwd=str(REPO), env=env, check=False
        )

        result["eval_return_code"] = eval_process.returncode

        if eval_process.returncode != 0:
            result["error"] = f"eval failed return_code={eval_process.returncode}"
            return result

        # ---------------- READ EVAL ----------------
        eval_file = result_dir / "eval.txt"

        if not eval_file.exists():
            result["error"] = "eval.txt not found"
            return result

        with open(eval_file, "r") as f:
            eval_data = json.load(f)

        result["test_loss_mean"] = eval_data["test"]["loss"]["mean"]
        result["test_loss_ci95"] = eval_data["test"]["loss"]["confidence"]
        result["test_acc_mean"] = eval_data["test"]["acc"]["mean"]
        result["test_acc_ci95"] = eval_data["test"]["acc"]["confidence"]

        result["success"] = True

    except Exception as e:
        result["error"] = repr(e)

    finally:
        result["elapsed_minutes"] = (time.time() - start) / 60

        print(f"\n[GPU {gpu_id}] EXP {exp_id:03d} DONE "
              f"({result['elapsed_minutes']:.1f} min)")

        if result["success"]:
            print(
                f"[GPU {gpu_id}] ACC = "
                f"{result['test_acc_mean'] * 100:.2f}% ± "
                f"{result['test_acc_ci95'] * 100:.2f}%"
            )
        else:
            print(f"[GPU {gpu_id}] FAILED: {result['error']}")

    return result


# ============================================================
# MANIFEST UPDATE (thread-safe)
# ============================================================

manifest_lock = threading.Lock()


def update_manifest(result):
    exp_id = result["experiment_id"]
    mask = manifest["experiment_id"] == exp_id

    with manifest_lock:
        manifest.loc[mask, "elapsed_minutes"] = result["elapsed_minutes"]
        manifest.loc[mask, "result_dir"] = result["result_dir"]
        manifest.loc[mask, "return_code"] = result["train_return_code"]
        manifest.loc[mask, "error"] = result["error"]

        if result["success"]:
            manifest.loc[mask, "train_status"] = "done"
            manifest.loc[mask, "eval_status"] = "done"
            manifest.loc[mask, "status"] = "done"

            manifest.loc[mask, "test_loss_mean"] = result["test_loss_mean"]
            manifest.loc[mask, "test_loss_ci95"] = result["test_loss_ci95"]
            manifest.loc[mask, "test_acc_mean"] = result["test_acc_mean"]
            manifest.loc[mask, "test_acc_ci95"] = result["test_acc_ci95"]

        else:
            manifest.loc[mask, "status"] = "failed"

            if result["train_return_code"] == 0:
                manifest.loc[mask, "train_status"] = "done"
                manifest.loc[mask, "eval_status"] = "failed"
            else:
                manifest.loc[mask, "train_status"] = "failed"

        manifest.to_csv(MANIFEST_PATH, index=False)


# ============================================================
# FIND PENDING EXPERIMENTS
# ============================================================

pending = manifest[
    (manifest["train_status"] != "done")
    | (manifest["eval_status"] != "done")
].copy()

# Hard resume guard: không bao giờ chạy lại EXP001-EXP051
pending = pending[pending["experiment_id"] >= START_EXP_ID].copy()
pending = pending.sort_values("experiment_id")

if MAX_EXPERIMENTS_THIS_RUN is not None:
    pending = pending.head(MAX_EXPERIMENTS_THIS_RUN)

print("=" * 90)
print("MULTI-JOB PAPER RUNNER (QUEUE)")
print("=" * 90)
print("Resume start            :", f"EXP{START_EXP_ID:03d}")
print("Skipped (backed-up)     :", START_EXP_ID - 1)
print("Pending selected        :", len(pending))

if len(pending) > 0:
    print(
        "Selected range          :",
        f"EXP{int(pending.iloc[0]['experiment_id']):03d}",
        "->",
        f"EXP{int(pending.iloc[-1]['experiment_id']):03d}",
    )

print("GPU IDs                 :", GPU_IDS)
print("Jobs per GPU            :", JOBS_PER_GPU)
print("Total parallel slots    :", len(SLOTS))
print("CPU threads per job     :", CPU_THREADS_PER_JOB)


# ============================================================
# QUEUE + WORKERS
# ============================================================

q = Queue()

for r in pending.to_dict("records"):
    q.put(r)


def worker(gpu_id):
    while True:
        try:
            row = q.get_nowait()
        except Empty:
            return

        result = run_experiment(row, gpu_id)
        update_manifest(result)


run_start = time.time()

if q.qsize() > 0:
    with ThreadPoolExecutor(max_workers=len(SLOTS)) as executor:
        futures = [executor.submit(worker, g) for g in SLOTS]
        for fut in futures:
            fut.result()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("RUN COMPLETE")
print("=" * 90)
print(f"Wall time this run: {(time.time() - run_start) / 60:.1f} min")

print("\nTrain status:")
print(manifest["train_status"].value_counts(dropna=False))

print("\nEval status:")
print(manifest["eval_status"].value_counts(dropna=False))

done = manifest[manifest["eval_status"] == "done"]

done_after_resume = manifest[
    (manifest["experiment_id"] >= START_EXP_ID)
    & (manifest["eval_status"] == "done")
]

overall_completed = min(
    len(manifest),
    (START_EXP_ID - 1) + len(done_after_resume),
)

print("\nCompleted in current manifest:", len(done), "/", len(manifest))
print("Overall progress incl. backed-up:", overall_completed, "/", len(manifest))

failed = manifest[manifest["status"] == "failed"]
if len(failed) > 0:
    print("\n⚠️ Failed experiments:")
    print(failed[["experiment_id", "way", "shot", "flag", "error"]].to_string(index=False))

if len(done) > 0:
    print("\nLatest completed results:")
    cols = [
        "experiment_id", "way", "shot", "flag",
        "test_acc_mean", "test_acc_ci95", "elapsed_minutes",
    ]
    print(done[cols].tail(10).to_string(index=False))

print("\nManifest:")
print(MANIFEST_PATH)

MULTI-JOB PAPER RUNNER (QUEUE)
Resume start            : EXP052
Skipped (backed-up)     : 51
Pending selected        : 24
Selected range          : EXP137 -> EXP160
GPU IDs                 : [0, 1]
Jobs per GPU            : 2
Total parallel slots    : 4
CPU threads per job     : 1

[GPU 0] START EXP 137
[GPU 0] 5-way 10-shot flag=0

[GPU 0] START EXP 138
[GPU 0] 5-way 10-shot flag=1

[GPU 1] START EXP 139
[GPU 1] 5-way 10-shot flag=2

[GPU 1] START EXP 140
[GPU 1] 5-way 10-shot flag=3


/kaggle/working/Few-Shot-KWS/scripts/train/few_shot/train.py:73: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  state['scheduler'].step()
Epoch 1 train:   0%|          | 0/200 [00:00<?, ?it/s]/kaggle/working/Few-Shot-KWS/scripts/train/few_shot/train.py:73: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  state[

Epoch 01: train loss = 0.689412, train acc = 0.748600, val loss = 0.659328, val acc = 0.780667
==> best model (loss = 0.659328), saving model...
Epoch 01: train loss = 0.818695, train acc = 0.693833, val loss = 1.072514, val acc = 0.593333
==> best model (loss = 1.072514), saving model...


Epoch 2 train:   0%|          | 0/200 [00:00<?, ?it/s]73.67it/s]

Epoch 01: train loss = 0.906125, train acc = 0.651200, val loss = 1.056675, val acc = 0.618667
==> best model (loss = 1.056675), saving model...


...

Epoch 200: train loss = 0.028653, train acc = 0.990143, val loss = 0.566355, val acc = 0.834381
Total Time: 0:16:40.379207


test: 100%|██████████| 100/100 [00:32<00:00,  3.12it/s]



[GPU 1] EXP 159 DONE (17.6 min)
[GPU 1] ACC = 86.76% ± 0.63%
Evaluating 5-way, 20-shot with 15 query examples/class over 100 episodes


test: 100%|██████████| 100/100 [00:30<00:00,  3.30it/s]



[GPU 1] EXP 160 DONE (17.5 min)
[GPU 1] ACC = 85.61% ± 0.67%

RUN COMPLETE
Wall time this run: 107.5 min

Train status:
train_status
done       109
pending     51
Name: count, dtype: int64

Eval status:
eval_status
done       109
pending     51
Name: count, dtype: int64

Completed in current manifest: 109 / 160
Overall progress incl. backed-up: 160 / 160

Latest completed results:
 experiment_id  way  shot  flag  test_acc_mean  test_acc_ci95  elapsed_minutes
           151    5    15     6       0.859333       0.006595        18.535827
           152    5    15     7       0.850762       0.006939        18.361423
           153    5    20     0       0.887333       0.007497        17.277308
           154    5    20     1       0.851467       0.008551        17.287903
           155    5    20     2       0.728111       0.008400        17.679956
           156    5    20     3       0.859667       0.007283        17.905431
           157    5    20     4       0.709111       0.008795 

In [20]:
# ============================================================
# BACKUP — RESULTS + MANIFEST
# ============================================================

import zipfile
import datetime
import pandas as pd
from pathlib import Path

RESULTS_SRC = Path("/kaggle/working/Few-Shot-KWS/results/paper_full_reproduction")
MANIFEST = Path("/kaggle/working/results/paper_full_reproduction/experiment_manifest.csv")

done_count = 0
if MANIFEST.exists():
    df = pd.read_csv(MANIFEST)
    done_count = int((df["eval_status"] == "done").sum())

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
zip_path = Path(f"/kaggle/working/paper_reproduction_{done_count}of160_{stamp}.zip")

print("Creating backup...")

n_files = 0

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:

    if RESULTS_SRC.exists():
        for p in RESULTS_SRC.rglob("*"):
            if p.is_file():
                z.write(p, Path("results") / p.relative_to(RESULTS_SRC))
                n_files += 1
    else:
        print("⚠️ Không thấy thư mục results:", RESULTS_SRC)

    if MANIFEST.exists():
        z.write(MANIFEST, "experiment_manifest.csv")
        n_files += 1
    else:
        print("⚠️ Không thấy manifest:", MANIFEST)

size_mb = zip_path.stat().st_size / 1024**2

print("\n" + "=" * 80)
print("BACKUP COMPLETE")
print("=" * 80)
print("File          :", zip_path)
print("Files in zip  :", n_files)
print("Size          :", round(size_mb, 2), "MB")
print("Done in manifest:", done_count, "/ 160")

Creating backup...

BACKUP COMPLETE
File          : /kaggle/working/paper_reproduction_109of160_20261002_1643.zip
Files in zip  : 642
Size          : 32.24 MB
Done in manifest: 109 / 160


In [21]:
import pandas as pd

p = "/kaggle/working/results/paper_full_reproduction/experiment_manifest.csv"
df = pd.read_csv(p)

print(df["train_status"].value_counts())
print(df["eval_status"].value_counts())
print("Completed:", (df["eval_status"] == "done").sum(), "/", len(df))

train_status
done       109
pending     51
Name: count, dtype: int64
eval_status
done       109
pending     51
Name: count, dtype: int64
Completed: 109 / 160
